# 1. process data

## 1.1 load packege

In [ ]:
import json
from collections import namedtuple, Counter

import numpy as np
import scanpy as sc
import pandas as pd
import squidpy as sq
import seaborn as sns

## Image manipulation and geometry
from tifffile import imread
from skimage.io import imread as skimread

## Plotting imports
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, Colormap
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib import rc_context

In [ ]:
### Functions for registration of Xenium to Visium data and associated analyis and visualization
## Visualization functions
from companion_functions import (
    hexlist_to_cmap,
    polygons_coords_to_patch_collection,
    plot_polygons,
    hex_corner_offset,
    polygon_corners,
    celltypes,
    celltypes,
    hex_codes,
    ctype_to_code_map,
    ctype_hex_map,
    ctype_cmap,
)

## Analysis functions
from companion_functions import (
    unique_encode,
    get_xenium_to_morphology_transform_from_xenium_morphology_image,
    get_xenium_capture_polygon_um,
    transform_coordinates,
    get_median_spot_to_spot_distance_from_centroids,
    generate_space_filling_visium_polygons,
    get_visium_capture_polygon,
    __OUTSIDE_VISIUM_CAPTURE_AREA_BARCODE__,
    bin_xenium_data_to_visium_spots,
    generate_anndata_matrix_from_transcript_assignments,
)

In [ ]:
import random
import numpy as np

# Set global seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)


## 1.2 load data and creat adata object

In [ ]:
import os
import scanpy as sc
import pandas as pd

def load_sample(sample_id, base_data_dir, base_roi_dir):
    """
    Load a sample and return the constructed AnnData object.
    sample_id: e.g. '29208'
    base_data_dir: Parent directory containing the H5 and CSV files
    base_roi_dir: Parent directory containing the ROI folders
    """
    print(f"\n>>> Processing sample {sample_id}")
    h5_path = os.path.join(base_data_dir, f"outs_{sample_id}", "cell_feature_matrix.h5")
    csv_path = os.path.join(base_data_dir, f"outs_{sample_id}", "cells.csv.gz")

    # Read H5 and CSV files
    adata = sc.read_10x_h5(h5_path)
    df_meta = pd.read_csv(csv_path)

    if 'cell_id' in df_meta.columns:
        df_meta.set_index('cell_id', inplace=True)
    
    # Check whether the indices match
    if not adata.obs_names.equals(df_meta.index):
        raise ValueError(f"[{sample_id}] Cell IDs do not match!")

    adata.obs = df_meta.copy()

    # Add spatial information
    if {'x_centroid', 'y_centroid'}.issubset(adata.obs.columns):
        adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy()
    else:
        raise ValueError(f"[{sample_id}] Missing 'x_centroid' or 'y_centroid'")

    # Add ROI information
    roi_path = os.path.join(base_roi_dir, sample_id)
    if os.path.isdir(roi_path):
        adata.obs['roi'] = 'other'
        roi_files = [f for f in os.listdir(roi_path) if f.endswith('_cells_stats.csv')]
        for file_name in roi_files:
            file_path = os.path.join(roi_path, file_name)
            roi_name = file_name.split('_cells_stats.csv')[0]

            try:
                df = pd.read_csv(file_path, header=None, sep=',', names=['Cell ID', 'Cluster', 'Transcripts', 'Area (µm^2)'])
                df = df[3:]  # Remove the first three rows of meta information
                cell_ids = df['Cell ID'].values

                match_count = 0
                for cid in cell_ids:
                    if cid in adata.obs_names:
                        adata.obs.at[cid, 'roi'] = roi_name
                        match_count += 1
                print(f"  - ROI {roi_name} matched cells: {match_count}")
            except Exception as e:
                print(f"  - Unable to read {file_name}: {e}")
    else:
        print(f"  - ROI folder not found: {roi_path}. Skipping ROI annotation.")

    print(f"  - Sample {sample_id} constructed successfully. Total cells: {adata.n_obs}")
    return adata


# ===== Main execution section =====
# Set the root directory of your data
base_data_dir = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\data\Xenium"
base_roi_dir = r"ROI"  # If the ROI folder is in the same directory as the code

# All sample IDs (from folder names)
sample_ids = ['29201', '29208', '41720', '53080', '53256']

# Construct all samples
adata_dict = {}
for sid in sample_ids:
    try:
        adata = load_sample(sid, base_data_dir, base_roi_dir)
        adata_dict[sid] = adata
    except Exception as e:
        print(f"Sample {sid} processing failed: {e}")

In [ ]:
import anndata as ad 
 
# Prepare sample list 
sample_ids = ['29201', '29208', '41720', '53080', '53256'] 
adatas_to_merge = [] 
 
for sample_id in sample_ids: 
    adata = adata_dict[sample_id].copy() 
 
    # Back up the original cell ID 
    adata.obs['cell_id_backup'] = adata.obs_names 
 
    # If roi does not exist, set it to 'other' 
    if 'roi' not in adata.obs.columns: 
        adata.obs['roi'] = 'other' 
 
    # Ensure roi is stored as string 
    adata.obs['roi'] = adata.obs['roi'].astype(str) 
 
    # Modify obs_names to make them unique 
    adata.obs_names = [f"{cid}-{sample_id}" for cid in adata.obs_names] 
 
    # Add batch information 
    adata.obs['batch'] = sample_id 
 
    adatas_to_merge.append(adata) 
 
# Merge using the new anndata.concat 
adata_combined = ad.concat( 
    adatas_to_merge, 
    axis=0, 
    join='inner',          # Keep only genes shared by all objects 
    label='batch',         # Specify a new obs column to indicate the source of each object 
    keys=sample_ids,       # Assign a label to each object; must match the order above 
    merge='same',          # Keep only identical .obs and .var columns (e.g. roi) 
    index_unique=None      # Do not automatically add -1, -2, etc.; we already set unique IDs manually 
) 
 
# Simple check 
print(f"Number of cells after merging: {adata_combined.n_obs}") 
print(f"Number of genes after merging: {adata_combined.n_vars}") 
print(f"Batch distribution:\n{adata_combined.obs['batch'].value_counts()}") 
print(f"ROI distribution:\n{adata_combined.obs['roi'].value_counts()}") 

In [ ]:
adata_combined.write("adata_merge_ovary_raw.h5ad")

## 1.3 QC

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
cprobes = (
    adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
cwords = (
    adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
print(f"Negative DNA probe count % : {cprobes}")
print(f"Negative decoding count % : {cwords}")


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata.obs["total_counts"][adata.obs["total_counts"]<3000],
    kde=False,
    ax=axs[0],
)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1],
)


axs[2].set_title("Area of segmented cells")
sns.histplot(
    adata.obs["cell_area"],
    kde=False,
    ax=axs[2],
)

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3],
)

In [ ]:
log_counts = np.log10(adata.obs['total_counts'] + 1)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Set style: use 'ticks' mode to remove the background grid and keep only the ticks
sns.set_style("ticks")

plt.figure(figsize=(4,3))

# The color code in your image is approximately #518EBA
target_blue = "#518EBA"

# 2. Plot histogram
# edgecolor='white' creates subtle gaps between bars for a cleaner appearance
sns.histplot(log_counts, bins=150, kde=True, color=target_blue, edgecolor='white', alpha=0.9)

# 3. Add title and labels
plt.title('Distribution of Total Counts (Log10 Scale)', fontsize=15, pad=15)
plt.xlabel('$Log_{10}(Total\ Counts + 1)$', fontsize=12)
plt.ylabel('Frequency (Cell Number)', fontsize=12)

# 4. Plot threshold lines
plt.axvline(x=np.log10(10), color='red', linestyle=':', linewidth=2.5, label='Min Threshold (10)')
plt.axvline(x=np.log10(233), color='blue', linestyle=':', linewidth=2.5, label='Peak (233)')

# 5. Optimize axes: show and thicken the left and bottom spines
ax = plt.gca()
for spine in ['left', 'bottom']:
    ax.spines[spine].set_color('black')
    ax.spines[spine].set_linewidth(1.5)

# Remove the top and right spines
sns.despine()

plt.legend(frameon=False)  # Remove the legend border

# 6. Save as a high-quality SVG
bbox_inches='tight',
plt.savefig(
    r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 1\Total_Counts_Distribution_cut_10_2st.svg",
    format='svg',
    bbox_inches='tight',
    dpi=300
)

plt.show()

In [ ]:
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=50)

## 1.4 Process fetal data

In [ ]:
fetal = adata[adata.obs["roi"].isin(["1_W22_22", "1_W16_16", "5_W9_9"])].copy()
fetal.layers['counts_raw']=fetal.X

In [ ]:
# Back up the original counts to the counts_raw layer
fetal.layers['counts_raw'] = fetal.X.copy()

# Ensure fetal is an independent object
fetal = fetal.copy()

# Check whether fetal is empty after filtering
if fetal.n_obs == 0:
    
    print("Warning: fetal is empty. Aborting processing.")
else:
    print(f"Info: fetal selected with {fetal.n_obs} cells.")
    
    # --- 2. Prepare raw count data (Raw Counts) ---
    if 'counts_raw' in fetal.layers:
        fetal.X = fetal.layers['counts_raw'].copy()
        print("Info: .X initialized from 'counts_raw' layer.")
    
    # --- 3. Basic preprocessing: normalization and log transformation ---
    sc.pp.normalize_total(fetal, inplace=True)
    sc.pp.log1p(fetal)
    
    # --- 4. Highly variable gene selection ---
    sc.pp.highly_variable_genes(fetal, n_top_genes=2000)
    
    # Back up the complete log-normalized data to .raw
    fetal.raw = fetal.copy()
    
    # --- 5. Dimensionality reduction and neighborhood graph calculation (using highly variable genes only) ---
    sc.pp.scale(fetal, max_value=10)
    sc.tl.pca(fetal, use_highly_variable=True)
    sc.pp.neighbors(fetal)
    sc.tl.umap(fetal, min_dist=0.01, spread=2)
    sc.tl.leiden(fetal, resolution=1, key_added='leiden_1')
    
    print(f"fetal (n={fetal.n_obs}) processing complete.")
    print(f"Total genes retained: {fetal.n_vars}")
    print(f"Highly variable genes used for analysis: {fetal.var.highly_variable.sum()}")

## 1.5 Process postnatal data

In [ ]:
remove_rois = [
    '1_W16_16',
    '3_W20_20',
    '8_W11_11',
    '7_W11_11',
    '14_LH7_2',
    '5_W9_9'
]

# 过滤，保留不在列表里的细胞
postnatal= adata[~adata.obs['roi'].isin(remove_rois)].copy()

In [ ]:
import scanpy as sc
import rapids_singlecell as rsc
import cupy as cp
import cupyx.scipy.sparse as cupsparse
import time
import gc

N_NEIGHBORS = 30
N_PCS = 50
UMAP_MIN_DIST = 0.3
UMAP_SPREAD = 1.0
LEIDEN_RESOLUTION = 0.5
RANDOM_STATE = 42

INPUT_FILE = "/exports/ovogrowth-hpc/fwei/prepubertal_ovary/adata_ovary_without_fetal_merge_after_QC_10.h5ad"
OUTPUT_FILE = INPUT_FILE.replace(".h5ad", "_rapids_clusters_only.h5ad")
CHECKPOINT_FILE = INPUT_FILE.replace(".h5ad", "_pca_checkpoint.h5ad")


# 1. Load data
print("[PY] [CPU] Loading postnatal data...")
postnatal = sc.read_h5ad(INPUT_FILE)
print(f"[PY] Loaded. Shape: {postnatal.shape}")

# 2. Highly variable genes on raw counts
# seurat_v3 works directly on count data
print("[PY] [CPU] Computing highly variable genes (seurat_v3, top 2000)...")
sc.pp.highly_variable_genes(
    postnatal,
    n_top_genes=2000,
    flavor='seurat_v3',
    subset=False
)

# 3. Normalize and log-transform before PCA
print("[PY] [CPU] Normalizing and log1p transforming...")
sc.pp.normalize_total(postnatal, target_sum=1e4)
sc.pp.log1p(postnatal)

# 4. PCA
# arpack is relatively memory-efficient for large sparse matrices
print("[PY] [CPU] Computing PCA (n_comps=50)...")
sc.pp.pca(
    postnatal,
    n_comps=50,
    use_highly_variable=True,
    svd_solver='arpack'
)

# 5. Save PCA checkpoint in case GPU steps fail
print("[PY] Saving PCA checkpoint...")
postnatal.write_h5ad(CHECKPOINT_FILE)
print(f"[PY] Checkpoint saved: {CHECKPOINT_FILE}")

# 6. Free expression matrix memory before moving to GPU
print("[PY] Freeing expression matrix from memory...")
del postnatal.X
gc.collect()

# 7. Move to GPU
print("[PY] Moving postnatal data to GPU...")
rsc.get.anndata_to_GPU(postnatal)

# 8. Compute neighbor graph based on PCA
print("[PY] [GPU] Computing neighbors (k=15, n_pcs=50)...")
rsc.pp.neighbors(
    postnatal,
    n_neighbors=15,
    n_pcs=50,
    use_rep='X_pca'
)

# 9. UMAP
print("[PY] [GPU] Computing UMAP...")
rsc.tl.umap(
    postnatal,
    min_dist=0.01,
    spread=2,
    random_state=42
)

# 10. Leiden clustering
print("[PY] [GPU] Leiden clustering (resolution=1)...")
rsc.tl.leiden(
    postnatal,
    resolution=0.55,
    key_added='leiden_rapids'
)

# 11. Move back to CPU
print("[PY] Moving postnatal data back to CPU...")

if hasattr(postnatal, "X") and isinstance(
    postnatal.X,
    (cp.ndarray, cupsparse.spmatrix)
):
    rsc.get.anndata_to_CPU(postnatal)

elif 'X_pca' in postnatal.obsm and isinstance(
    postnatal.obsm['X_pca'],
    cp.ndarray
):
    for key in postnatal.obsm.keys():
        if isinstance(postnatal.obsm[key], cp.ndarray):
            postnatal.obsm[key] = postnatal.obsm[key].get()

# 12. Save final output
print(f"[PY] Saving output: {OUTPUT_FILE}")
postnatal.write_h5ad(OUTPUT_FILE)

duration = time.time() - start_time
print(f"[SUCCESS] Done! Total time: {duration/60:.2f} minutes")

In [ ]:
postnatal.obs["leiden_0.55"] = postnatal.obs["leiden_rapids"]

n_per_cluster = 10000
idx = []
np.random.seed(42)

for cluster in postnatal.obs['leiden_0.55'].unique():
    cluster_idx = np.where(postnatal.obs['leiden_0.55'] == cluster)[0]
    if len(cluster_idx) > n_per_cluster:
        idx.extend(np.random.choice(cluster_idx, n_per_cluster, replace=False))
    else:
        idx.extend(cluster_idx)

postnatal_sub = postnatal[idx].copy()
print(f"Number of cells after downsampling: {postnatal_sub.n_obs}")

postnatal_sub.layers['counts_raw'] = postnatal_sub.X.copy()

sc.pp.normalize_total(postnatal_sub)
sc.pp.log1p(postnatal_sub)

sc.pp.highly_variable_genes(
    postnatal_sub,
    n_top_genes=2000,
    batch_key='batch',
    subset=False
)

sc.pp.scale(postnatal_sub)

sc.tl.pca(
    postnatal_sub,
    n_comps=15,
    use_highly_variable=True
)

sc.pp.neighbors(
    postnatal_sub,
    use_rep='X_pca',
    n_neighbors=15,
    random_state=42
)

sc.tl.umap(postnatal_sub, random_state=42)

sc.pl.umap(postnatal_sub, color=['leiden_0.55'])

# 2 Figure 1

## 2.1 Figure 1e

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---- data (S29 corrected: custom=86, 10x=6) ----
sections = ["S9", "S19", "S29"]
methods  = ["Manual", "10x pipeline", "Custom"]
counts = {
    "Manual":       [77, 37, 83],
    "10x pipeline": [11,  2,  6],
    "Custom":       [79, 43, 86],
}

# ---- style ----
colors = {"Manual": "#2E86DE", "10x pipeline": "#FF9F1C", "Custom": "#E4007C"}
x = np.arange(len(sections))
width = 0.26

fig, ax = plt.subplots(figsize=(4, 4))
for i, m in enumerate(methods):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, counts[m], width, label=m,
                  color=colors[m], edgecolor="black", linewidth=0.6)
    ax.bar_label(bars, padding=2, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(sections)
ax.set_ylabel("Number of oocytes detected")
ax.set_xlabel("Section")
ax.set_title("Oocyte detection by segmentation method", fontsize=11)
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.margins(y=0.12)

plt.tight_layout()
plt.savefig(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\2st_2026_06_09\Figure 1\oocyte_detection_barplot.pdf", bbox_inches="tight")
plt.show()

## 2.2 Figure 1f

In [ ]:
custom_palette = {
   # Low-quality
    "0":  "#D9D9D9",   # LQC1
    "15": "#BDBDBD",   # LQC2

    # Stromal / somatic
    "1":  "#E69F00",   # aStr
    "3":  "#0072B2",   # tStr
    "11": "#009E73",   # eStr
    "10": "#8E6BBE",   # MC / mural cells
    "16": "#FFD92F",   # sSC

    # Pre-granulosa / OSE
    "2":  "#00C853",   # preGC1
    "6":  "#00E5FF",   # preGC2
    "12": "#A65628",   # OSE

    # Germ-cell lineage
    "4":  "#E41A1C",   # OOG
    "5":  "#FF00A8",   # OGC
    "8":  "#FF7F00",   # SGC
    "9":  "#0057FF",   # PGC

    # Vascular / lymphatic / immune
    "7":  "#1F78B4",   # VEC
    "13": "#6B6B6B",   # Mac
    "14": "#00A6A6",   # LEC
}

In [ ]:

umap_coords = fetal.obsm["X_umap"]

leiden_clusters = fetal.obs["leiden_1"]

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


umap_df = pd.DataFrame(
    umap_coords,
    columns=["UMAP_1", "UMAP_2"],
    index=fetal.obs_names
)

umap_df["leiden_1"] = leiden_clusters.astype(str).values


plt.figure(figsize=(7, 5))

sns.scatterplot(
    data=umap_df,
    x="UMAP_1",
    y="UMAP_2",
    hue="leiden_1",
    palette=custom_palette,
    s=5,
    edgecolor=None
)

plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")

plt.legend(
    title="Leiden Cluster",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    markerscale=3
)

plt.tight_layout()

plt.savefig(
    r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\2st_2026_06_09\Figure 1\UMAP\umap_fetal_res1.png",
    format="png",
    dpi=900,
    bbox_inches="tight"
)

plt.show()

## 2.2 Figure 1g

In [ ]:
my_genes=[
    "PDGFRB","POSTN","MEG3","CYP17A1", "CD14","CD34","FLT4", "SOX2-OT","NLRP5","PDIA3","DAZL","POU5F1","KITLG","FOXL2","GATA4","SFRP1"
]
sc.pl.dotplot(
    fetal,
    var_names=my_genes,
    groupby='leiden_1',              
    standard_scale='var',          
    cmap='Reds',
    dendrogram=True,               
    figsize=(7.5, 3.8),
    save='fetal_clusters_res1_3st.pdf'
)

## 2.3 Figure 1i

In [ ]:
custom_palette = {
    "0": "#00ff7f",
    "1": "#ffd700",
    "2": "#ff8c00",
    "3": "#00FFFF",
    "4": "#ff00ff",
    "5": "#32cd32",
    "6": "#00ced1",
    "7": "#ff4500",
    "8": "#ff69b4",
    "9": "#ff002a",
    "10": "#565dfd",
    "11": "#585b5c",
    "12": "#1e90ff",
    "13": "#ADFF2F",
}

In [ ]:

umap_coords =postnatal_sub.obsm["X_umap"]

leiden_clusters =postnatal_sub.obs["leiden_0.55"]

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


umap_df = pd.DataFrame(
    umap_coords,
    columns=["UMAP_1", "UMAP_2"],
    index=fetal.obs_names
)

umap_df["leiden_0.55"] = leiden_clusters.astype(str).values


plt.figure(figsize=(7, 5))

sns.scatterplot(
    data=umap_df,
    x="UMAP_1",
    y="UMAP_2",
    hue="leiden_0.55",
    palette=custom_palette,
    s=5,
    edgecolor=None
)

plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")

plt.legend(
    title="Leiden Cluster",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    markerscale=3
)

plt.tight_layout()

plt.savefig(
    r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\2st_2026_06_09\Figure 1\UMAP\umap_postnal_res0.55.png",
    format="png",
    dpi=900,
    bbox_inches="tight"
)

plt.show()

In [ ]:
## 2.4 Figure 1j

In [ ]:
my_genes=[
    "EPAS1","SOX2-OT","NR4A1","ADAMTS1", "EEF1G","CD22","PDGFRA", "TUBB","TIMP2","CD14","HSD17B1","DAZL","STAR","GJA1"
]
sc.pl.dotplot(
    postnatal_sub,
    var_names=my_genes,
    groupby='leiden_1',  
    standard_scale='var',  
    #vmax=2, 
    cmap="Reds",
    figsize=(7.5, 3.8),
    dendrogram=True,
    save='postnal_leiden_sub_normalize.pdf'
)

# 3 Fig.2 

In [ ]:
SC5=sc.read_h5ad((r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\adata\5st_12st_Figure2_5st\C5_sub_8st.h5ad"))

In [ ]:
SC5.obs["leiden_final"]= SC5.obs["leiden_sub"].copy()

In [ ]:
import pandas as pd

fetal.obs["leiden_c5"] = fetal.obs["leiden_1"].astype(str)

c5_map = {
    "0": "5.0",
    "1": "5.1",
    "2": "5.2",
    "3": "5.3",
}

sc5_labels = (
    SC5.obs["leiden_final"]
    .astype(str)
    .map(c5_map)
)

print("Unmapped labels in SC5:")
print(SC5.obs["leiden_final"].astype(str)[sc5_labels.isna()].unique())

common_cells = fetal.obs_names.intersection(SC5.obs_names)

print(f"Matched cells: {len(common_cells)}")

fetal.obs.loc[common_cells, "leiden_c5"] = sc5_labels.loc[common_cells]

fetal.obs["leiden_c5"] = fetal.obs["leiden_c5"].astype("category")

print(fetal.obs["leiden_c5"].value_counts().sort_index())

In [ ]:
adata_sub = fetal[fetal.obs["leiden_c5"].astype(str).isin(["9","4", "8","5.1","5.2"])].copy()

if adata_sub.n_obs == 0:
    print("Warning: adata_sub is empty. Aborting processing.")
else:
    print(f"Info: adata_sub selected with {adata_sub.n_obs} cells.")
    
    if 'counts_raw' in adata_sub.layers:
        adata_sub.X = adata_sub.layers['counts_raw'].copy()
        print("Info: .X initialized from 'counts_raw' layer.")
    
    sc.pp.normalize_total(adata_sub, inplace=True)
    sc.pp.log1p(adata_sub)
    sc.pp.highly_variable_genes(adata_sub, n_top_genes=2000)
    adata_sub.raw = adata_sub.copy()
    sc.pp.scale(adata_sub, max_value=10)
    sc.tl.pca(adata_sub, use_highly_variable=True)
    sc.pp.neighbors(adata_sub)
    sc.tl.umap(adata_sub, min_dist=0.01, spread=2)
    sc.tl.leiden(adata_sub, resolution=1, key_added='leiden_1.1')
    
    print(f"adata_sub (n={adata_sub.n_obs}) processing complete.")
    print(f"Total genes retained: {adata_sub.n_vars}")
    print(f"Highly variable genes used for analysis: {adata_sub.var.highly_variable.sum()}")

In [ ]:
sc.pl.umap(adata_sub,color=['leiden_1.1'])

In [ ]:
genes_from_figure= [
    "POU5F1", "KIT", "PDPN", "FGFR3", 
    "DAZL", "PRAME", "MYBL2", "MCM4", "HELLS"
    "REC8", "RAD21L1", "HORMAD1", "SPATA22", "MEIOB", "RAD51","FOXL2","KITLG","GATA4","IGFBP2"
    "SYCP1", "SYCE2", "BRDT", "CCNB3","SYCP1",
    "TP63","NLRP5", "NLRP2", "NLRP9", "GDF9", "WEE2", "TLE6"
]
genes_in_adata = [g for g in genes_from_figure if g in adata_sub.var_names]

sc.pl.dotplot(
    adata_sub,
    var_names=genes_in_adata,
    groupby='leiden_1.1',              
    standard_scale='var',          
    cmap='Reds',
    dendrogram=True,              
    figsize=(10, 6),
    #save='fetal_19clusters_annotation.pdf'
)

In [ ]:
# remove mixed granulosa cell and unknow cell
adata_sub = adata_sub[~adata_sub.obs['leiden_1.1'].astype(str).isin(['11',"3"])].copy()


adata_sub.obs['leiden_1.1'] = (
    adata_sub.obs['leiden_1.1'].astype(str).astype('category')
    .cat.remove_unused_categories()
)

In [ ]:
sc.tl.pca(adata_sub, use_highly_variable=True)
sc.pp.neighbors(adata_sub)
sc.tl.umap(adata_sub, min_dist=0.01, spread=2)
sc.tl.leiden(adata_sub, resolution=1, key_added='leiden_1.2')

## 3.1 Fig.2a

In [ ]:
custom_palette = {
    "0": "#00ff7f",
    "1": "#ffd700",
    "2": "#ff8c00",
    "3": "#00FFFF",
    "4": "#ff00ff",
    "5": "#32cd32",
    "6": "#00ced1",
    "7": "#ff4500",
    "8": "#ff69b4",
    "9": "#ff002a",
    "10": "#565dfd",
    "11": "#585b5c",
    "12": "#1e90ff",
    "13": "#ADFF2F",
}

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

umap_coords = adata_sub.obsm["X_umap"]

leiden_clusters = adata_sub.obs["leiden_1.2"].astype(str)

umap_df = pd.DataFrame(
    umap_coords,
    columns=["UMAP_1", "UMAP_2"],
    index=adata_sub.obs_names
)

umap_df["leiden_1.2"] = leiden_clusters.values

print("Clusters in leiden_final:")
print(umap_df["leiden_1.2"].value_counts().sort_index())

print("Palette keys:")
print(custom_palette.keys())

plt.figure(figsize=(6, 4))

sns.scatterplot(
    data=umap_df,
    x="UMAP_1",
    y="UMAP_2",
    hue="leiden_1.2",
    palette=custom_palette,
    s=3,
    edgecolor=None,
    linewidth=0
)

plt.title("UMAP Plot with Leiden Final Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")

plt.legend(
    title="Leiden 1.2",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    markerscale=5,
    frameon=False
)

plt.tight_layout()

plt.savefig(
    r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\umap\umap_germ_celline_res1_2.png",
    format="png",
    dpi=900,
    bbox_inches="tight"
)

plt.show()

## 3.2 Fig.2b

In [ ]:
genes_from_figure= ["FGFR3","TLE6","NLRP5","TP63","DAZL","STRA8","POU5F1", "KIT","NANOG","MYBL2","PRAME","PAIP2","SPATA22", "MEIOB","CCNB3","SYCP1","BRDT"]
genes_in_adata = [g for g in genes_from_figure if g in adata_sub.var_names]

In [ ]:
sc.pl.dotplot(
    adata_sub,
    var_names=genes_in_adata,
    groupby='leiden_1.2',              
    standard_scale='var',         
    cmap='Reds',
    dendrogram=True,            
    figsize=(6, 3),
    save='fetal_germ_cell_line.pdf'
)

## 3.3 Fig.2c

In [ ]:
cluster_key = "leiden_1.2"

roi_map = {
    "5_W9_9": "W9",
    "1_W16_16": "W16",
    "1_W22_22": "W22",
}

roi_order = ["W9", "W16", "W22"]
remove_clusters = []

obs_plot = adata_sub.obs[["roi", cluster_key]].copy()
obs_plot[cluster_key] = obs_plot[cluster_key].astype(str)

obs_plot = obs_plot[
    obs_plot["roi"].isin(roi_map.keys()) &
    (~obs_plot[cluster_key].isin(remove_clusters))
].copy()

obs_plot["roi_stage"] = obs_plot["roi"].map(roi_map)
obs_plot["roi_stage"] = pd.Categorical(
    obs_plot["roi_stage"],
    categories=roi_order,
    ordered=True
)

cross_tab = pd.crosstab(
    obs_plot[cluster_key],
    obs_plot["roi_stage"]
)

cross_tab = cross_tab[roi_order]

cluster_order = sorted(
    cross_tab.index,
    key=lambda x: int(x)
)

cross_tab = cross_tab.loc[cluster_order]

roi_totals = cross_tab.sum(axis=0)

cross_tab_prop = cross_tab.div(
    roi_totals,
    axis=1
)

plot_data = cross_tab_prop.T
plot_data = plot_data[plot_data.columns[::-1]]

n_clusters = len(plot_data.columns)
cmap = plt.cm.tab20 if n_clusters <= 20 else plt.cm.gist_ncar

color_list = [
    cmap(i / max(n_clusters - 1, 1))
    for i in range(n_clusters)
]

fig, ax = plt.subplots(figsize=(5.5, 6))

plot_data.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=custom_palette,
    width=0.8,
    legend=False
)

for i, roi in enumerate(roi_order):
    total = roi_totals.loc[roi]
    ax.text(
        i,
        1.02,
        f"N={int(total)}",
        ha="center",
        va="bottom",
        fontsize=10,
        rotation=45
    )

plt.title(
    f"Proportion of fetal clusters across ROIs ({cluster_key})",
    fontsize=12,
    pad=20
)

plt.xlabel("")
plt.ylabel("Proportion", fontsize=12)

ax.set_ylim(0, 1.15)

plt.xticks(rotation=0)

handles, labels = ax.get_legend_handles_labels()
handles, labels = handles[::-1], labels[::-1]

ax.legend(
    handles,
    labels,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8,
    frameon=False,
    title="Cluster",
    title_fontsize=9
)

plt.tight_layout()

plt.savefig(
    r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\barplot\umap_germ_celline_c5_res1_2.pdf",
    format="pdf",
    dpi=900,
    bbox_inches="tight"
)

plt.show()

In [ ]:
adata_sub.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\adata\5st_12st_Figure2_5st\germ_cellline_8st.h5ad")

## 3.4 Fig.2d

In [ ]:
import squidpy as sq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
adata_sub.obs['leiden_1']= adata_sub.obs['leiden_1.2'].copy()
adata_spatial = adata_sub[adata_sub.obs['roi'] == '1_W16_16']
keep = ["0", "1", "2", "3", "4", "7", "6"]
adata_spatial = adata_spatial[
    adata_spatial.obs['leiden_1'].astype(str).isin(keep)
].copy()

adata_spatial.obsm["spatial"] = (
    adata_spatial.obs[["x_centroid", "y_centroid"]]
    .astype(float)
    .to_numpy()
)

valid_cells = np.isfinite(
    adata_spatial.obsm["spatial"]
).all(axis=1)

adata_spatial = adata_spatial[valid_cells].copy()

adata_spatial.obs["leiden_1"] = (
    adata_spatial.obs["leiden_1"]
    .astype(str)
    .astype("category")
)

sq.gr.spatial_neighbors(
    adata_spatial,
    coord_type="generic",
    n_neighs=30,
    spatial_key="spatial"
)

sq.gr.nhood_enrichment(
    adata_spatial,
    cluster_key="leiden_1"
)

sq.pl.nhood_enrichment(
    adata_spatial,
    cluster_key="leiden_1",
    method="average",
    cmap="RdBu_r",
    vmin=-100,
    vmax=50,
    figsize=(12, 10),
    show=False
)

plt.tight_layout()

plt.savefig(
    "adata_sub_nhood_enrichment.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

adata_plot = adata_spatial
cluster_key = "leiden_1"

uns_key = f"{cluster_key}_nhood_enrichment"

if uns_key not in adata_plot.uns:
    raise KeyError(
        f"{uns_key!r} is not in adata_plot.uns. Please run:\n"
        f"sq.gr.nhood_enrichment(adata_plot, cluster_key='{cluster_key}')"
    )

z = adata_plot.uns[uns_key]["zscore"].astype(float).copy()
z = (z + z.T) / 2
z[~np.isfinite(z)] = np.nan

orig_labels = (
    adata_plot.obs[cluster_key]
    .cat.categories
    .astype(str)
    .tolist()
)

assert z.shape[0] == len(orig_labels), (
    f"Matrix shape {z.shape} does not match the number of categories "
    f"{len(orig_labels)}.\n"
    "The cluster categories may have been modified after enrichment analysis.\n"
    "Please rerun spatial_neighbors and nhood_enrichment before plotting."
)

df = pd.DataFrame(
    z,
    index=orig_labels,
    columns=orig_labels
)

developmental_order = [
    "9", "0", "2", "3", "4", "7", "8", "5", "6", "10", "1"
]

labels = [
    lb for lb in developmental_order
    if lb in orig_labels
]

remaining = [
    lb for lb in orig_labels
    if lb not in labels
]

labels = labels + sorted(
    remaining,
    key=lambda x: float(x)
)

df = df.reindex(
    index=labels,
    columns=labels
)

mat = df.to_numpy()
n = len(labels)

cluster_names = {
    "9": "Early PGC",
    "0": "Mitotic PGC",
    "2": "Late premeiotic",
    "3": "Leptotene",
    "4": "Zygotene",
    "7": "Pachytene",
    "8": "Pachytene–diplotene",
    "6": "Early dictyate",
    "10": "Late dictyate",
    "11": "FGFR3-high",
    "1": "Low-transcript",
    "5": "FGFR3-high dictyate",
}

display_labels = [
    f"{lb}  {cluster_names.get(lb, '')}".strip()
    for lb in labels
]

vmin, vmax = -50, 30

if "custom_palette" not in globals():
    color_key = f"{cluster_key}_colors"

    cat_order = (
        adata_plot.obs[cluster_key]
        .cat.categories
        .astype(str)
        .tolist()
    )

    if color_key in adata_plot.uns:
        custom_palette = dict(
            zip(cat_order, adata_plot.uns[color_key])
        )
    else:
        fb = plt.cm.tab20(
            np.linspace(0, 1, len(cat_order))
        )
        custom_palette = dict(
            zip(cat_order, fb)
        )

cmap = plt.cm.RdBu_r.copy()
cmap.set_bad("#f0f0f0")

fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(
    mat,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    aspect="equal",
    interpolation="none"
)

ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))

ax.set_xticklabels(
    display_labels,
    fontsize=10,
    rotation=90
)

ax.set_yticklabels(
    display_labels,
    fontsize=10
)

for k in range(1, n):
    ax.axhline(
        k - 0.5,
        color="white",
        linewidth=0.6
    )
    ax.axvline(
        k - 0.5,
        color="white",
        linewidth=0.6
    )

for i, lb in enumerate(labels):
    ax.add_patch(
        mpatches.Rectangle(
            (-1.75, i - 0.5),
            0.65,
            1.0,
            facecolor=custom_palette.get(
                str(lb),
                "#808080"
            ),
            edgecolor="none",
            clip_on=False,
            transform=ax.transData
        )
    )

for j, lb in enumerate(labels):
    ax.add_patch(
        mpatches.Rectangle(
            (j - 0.5, -1.75),
            1.0,
            0.65,
            facecolor=custom_palette.get(
                str(lb),
                "#808080"
            ),
            edgecolor="none",
            clip_on=False,
            transform=ax.transData
        )
    )

ax.set_xlim(-1.9, n - 0.5)
ax.set_ylim(n - 0.5, -1.9)

cbar = fig.colorbar(
    im,
    ax=ax,
    shrink=0.65,
    pad=0.06
)

cbar.set_label(
    "Neighborhood enrichment Z-score",
    fontsize=11
)

cbar.ax.tick_params(
    labelsize=9
)

ax.set_title(
    "Spatial neighborhood enrichment of fetal germ-cell states",
    fontsize=13,
    pad=28
)

ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()

plt.savefig(
    r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\neibohoud\neibohoud_W16.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
adata_sub.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\adata\5st_12st_Figure2_5st\germ_cellline_8st.h5ad")

## 3.5 Fig.2e

In [ ]:
import anndata as ad
import squidpy as sq
import cellcharter as cc
import pandas as pd
import scanpy as sc
import scvi
import numpy as np
import matplotlib.pyplot as plt
scvi.settings.seed = 12345

In [ ]:
adata_w16 = fetal[fetal.obs['roi'] == '1_W16_16'].copy()

In [ ]:
adata_w16.obs["leiden_c5"] = adata_w16.obs["leiden_1"].astype(str)

c5_map = {
    "0": "5.0",
    "1": "5.1",
    "2": "5.2",
    "3": "5.3",
    "4": "5.4",
    "5": "5.5",
    "6": "5.6",
    "7": "5.7",
    "8": "5.8",
    "9": "5.9",
    "10": "5.10",
}

sc5_labels = adata_sub.obs["leiden_1.2"].astype(str).map(c5_map)

unmapped = adata_sub.obs["leiden_1.2"].astype(str)[sc5_labels.isna()].unique()
print("Unmapped labels in SC5:", unmapped)

common_cells = adata_w16.obs_names.intersection(adata_sub.obs_names)
print(f"Matched cells: {len(common_cells)}")

sub = sc5_labels.loc[common_cells].dropna()
adata_w16.obs.loc[sub.index, "leiden_c5"] = sub

adata_w16.obs["leiden_c5"] = adata_w16.obs["leiden_c5"].astype("category")

print(
    adata_w16.obs["leiden_c5"]
    .value_counts()
    .sort_index()
)

In [ ]:
adata = adata_w16.copy()

# =========================
# 0) Input checks
# =========================
assert "counts_raw" in adata.layers.keys(), "layers['counts_raw'] not found"
spatial_key = "spatial"   # 可改 "spatial" / "spatial_spline_aligned"
assert spatial_key in adata.obsm_keys(), f"{spatial_key} not in adata.obsm"

use_batch = False
batch_key = "batch"
if use_batch:
    assert batch_key in adata.obs.columns
    adata.obs[batch_key] = adata.obs[batch_key].astype("category")

# =========================
# 1) Filtering on raw counts
# =========================
adata.X = adata.layers["counts_raw"].copy()


# =========================
# 2) Save raw counts for scVI  (tutorial-style)
# =========================
adata.layers["counts"] = adata.X.copy()

# =========================
# 3) Normalize/log for QC & plotting (NOT for scVI)
# =========================
sc.pp.normalize_total(adata, target_sum=1e6)
sc.pp.log1p(adata)

# =========================
# 4) Dimensionality reduction (scVI)  
# =========================
setup_kwargs = dict(layer="counts")
if use_batch and adata.obs[batch_key].nunique() > 1:
    setup_kwargs["batch_key"] = batch_key

scvi.model.SCVI.setup_anndata(adata, **setup_kwargs)

model = scvi.model.SCVI(adata, n_latent=10)   
model.train(early_stopping=True, enable_progress_bar=True)

adata.obsm["X_scVI"] = model.get_latent_representation().astype(np.float32)

# =========================
# 5) Spatial graph (neighbors) for CellCharter
# =========================
sq.gr.spatial_neighbors(
    adata,
    coord_type="generic",
    delaunay=True,
    spatial_key=spatial_key,
    percentile=99,
    library_key=batch_key if (use_batch and adata.obs[batch_key].nunique() > 1) else None,
)
cc.gr.remove_long_links(adata)



In [ ]:
# =========================
# 6) CellCharter: neighborhood aggregation (core step)
# =========================
cc.gr.aggregate_neighbors(
    adata,
    n_layers=3,
    use_rep="X_scVI",
    out_key="X_cellcharter",
    sample_key=batch_key if (use_batch and adata.obs[batch_key].nunique() > 1) else None,
)

# =========================
# 7) CellCharter’s spatial clustering (AutoK -> Cluster)
# =========================
autok = cc.tl.ClusterAutoK(n_clusters=(2, 12), max_runs=10)
autok.fit(adata, use_rep="X_cellcharter")
cc.pl.autok_stability(autok)

In [ ]:
K = 6
clusterer = cc.tl.Cluster(n_clusters=K)
clusterer.fit(adata, use_rep="X_cellcharter")

adata.obs["cluster_cellcharter_6"] = clusterer.predict(   
    adata, use_rep="X_cellcharter"
).astype(str)

print(adata.obs["cluster_cellcharter_6"].value_counts())  
adata.uns.pop('cluster_cellcharter_colors', None)  
sq.pl.spatial_scatter(
    adata,
    color="cluster_cellcharter_6",
    spatial_key="spatial",
    shape=None,          
    size=15,
    figsize=(15, 15),
)

In [ ]:
cc.gr.enrichment(
    adata,
    group_key="cluster_cellcharter_6",   
    label_key="leiden_c4",             
)

cc.pl.enrichment(
    adata,
    group_key="cluster_cellcharter_6",
    label_key="leiden_c4",
    figsize=(8, 6),
    save=r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\cellcharter_enrichment.pdf"
)

In [ ]:
adata.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\adata\adata_w16_cellcharter.h5ad")

## 3.6 Fig.2h

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import matplotlib.ticker as mticker

# ============================================================
# Fully enhanced version: plot gene expression curves along the AP axis
# Supports subset filtering and dual Y-axes
# ============================================================
def plot_genes_along_AP(
    adata,                          # AnnData object
    gene_list=None,                 # Gene list (simple mode)
    gene_configs=None,              # Gene configuration list (advanced mode)
    coord_col='AP',                 # Coordinate column name
    subset_col='sub_leiden',        # Column used for subset filtering
    n_bins=50,
    show_points=True,
    smooth=0,
    figsize=(12, 6),
    linewidth=2,
    fill=False,
    fill_alpha=0.3,                 # Fill transparency
    gene_colors=None,
    gene_linestyles=None,
    vertical_lines=None,
    vline_color='blue',
    vline_style='--',
    vline_width=1.5,
    vline_alpha=0.8,
    show_legend=True,
    xlim=None,                      # X-axis range, e.g. (0, 2500)
    ylim=None,                      # Primary Y-axis range, e.g. (0, 1)
    x_tick_interval=100,            # X-axis tick interval
    secondary_ylim=None,            # Secondary Y-axis range
    normalization='minmax'          # Normalization method: 'minmax', 'zscore', 'none'
):
    """
    Plot gene expression curves along a specified coordinate axis.
    Supports subset filtering and dual Y-axes.

    Parameters
    ----------
    Basic parameters
    ----------------
    adata : AnnData
        AnnData object.

    gene_list : list
        List of gene names for simple mode.
        All genes use the same settings.

    gene_configs : list of dict
        Gene configuration list for advanced mode.
        Each element should be a dictionary in the following format:

        {
            'genes': ['OTX2', 'EN1'],
            'subset_values': [0, 1, 2, 3],
            'use_secondary_y': False,
            'linestyle': '-',
            'color': None,
            'n_bins': None,
        }

        Example:

        gene_configs = [
            {
                'genes': ['OTX2', 'EN1', 'GBX2'],
                'subset_values': list(range(13)),
                'use_secondary_y': False,
                'linestyle': '-',
                'n_bins': 50
            },
            {
                'genes': ['PAX6', 'SOX2'],
                'subset_values': list(range(13)),
                'use_secondary_y': True,
                'linestyle': '--',
                'n_bins': 30
            }
        ]

    coord_col : str
        Coordinate column name. Default is 'AP'.

    subset_col : str
        Column used for subset filtering.
        Default is 'sub_leiden'.

    Plotting parameters
    -------------------
    n_bins : int
        Number of bins along the coordinate axis.
        Default is 50.

    show_points : bool
        Whether to display data points.
        Default is True.

    smooth : int
        Curve smoothing level.
        0 means no smoothing.
        Recommended range: 0-5.

    figsize : tuple
        Figure size.
        Default is (12, 6).

    linewidth : float
        Line width.
        Default is 2.

    fill : bool
        Whether to fill the area under the curve.
        Default is False.

    fill_alpha : float
        Fill transparency.
        Default is 0.3.

    normalization : str
        Normalization method:
        - 'minmax': normalize to [0, 1]
        - 'zscore': Z-score normalization
        - 'none': no normalization

    Style parameters for simple mode
    --------------------------------
    gene_colors : dict
        Gene color dictionary: {gene: color}

    gene_linestyles : dict
        Gene linestyle dictionary: {gene: linestyle}

    Reference line parameters
    -------------------------
    vertical_lines : list
        Positions of vertical reference lines.

    vline_color : str
        Vertical line color.
        Default is 'blue'.

    vline_style : str
        Vertical line style.
        Default is '--'.

    vline_width : float
        Vertical line width.
        Default is 1.5.

    vline_alpha : float
        Vertical line transparency.
        Default is 0.8.

    Other parameters
    ----------------
    show_legend : bool
        Whether to display the legend.
        Default is True.

    xlim : tuple
        X-axis range, e.g. (0, 2500).

    ylim : tuple
        Primary Y-axis range, e.g. (0, 1).

    x_tick_interval : float
        X-axis tick interval.
        Default is 100.

    secondary_ylim : tuple
        Secondary Y-axis range, e.g. (0, 1).

    Returns
    -------
    fig
        Matplotlib figure object.

    gene_curves
        Dictionary containing the expression curve data for each gene.
    """

    # ============================================================
    # 1. Process input parameters
    # ============================================================
    if gene_configs is None and gene_list is None:
        raise ValueError("Either gene_list or gene_configs must be provided.")

    if coord_col not in adata.obs.columns:
        raise KeyError(f"Coordinate column '{coord_col}' was not found in adata.obs.")

    if subset_col not in adata.obs.columns:
        raise KeyError(f"Subset column '{subset_col}' was not found in adata.obs.")

    if gene_configs is None:
        gene_configs = [{
            'genes': gene_list,
            'subset_values': None,
            'use_secondary_y': False,
            'linestyle': None,
            'color': None
        }]

    # ============================================================
    # 2. Create figure and axes
    # ============================================================
    fig, ax = plt.subplots(figsize=figsize)

    need_secondary_y = any(
        config.get('use_secondary_y', False)
        for config in gene_configs
    )

    ax2 = None

    if need_secondary_y:
        ax2 = ax.twinx()

    # ============================================================
    # 3. Process each configuration group
    # ============================================================
    all_gene_curves = {}

    for config in gene_configs:

        genes = config['genes']
        subset_values = config.get('subset_values', None)
        use_secondary_y = config.get('use_secondary_y', False)
        config_linestyle = config.get('linestyle', '-')
        config_color = config.get('color', None)
        config_n_bins = config.get('n_bins', n_bins)

        current_ax = (
            ax2
            if (use_secondary_y and ax2 is not None)
            else ax
        )

        if subset_values is not None:

            subset_values_str = [
                str(v) for v in subset_values
            ]

            mask = (
                adata.obs[subset_col]
                .astype(str)
                .isin(subset_values_str)
            )

            adata_subset = adata[mask, :].copy()

            print(
                f"Using cells with {subset_col} values "
                f"{subset_values}: {mask.sum()} cells, "
                f"bins={config_n_bins}"
            )

        else:

            adata_subset = adata

            print(
                f"Using all cells: "
                f"{adata_subset.n_obs} cells, "
                f"bins={config_n_bins}"
            )

        coords = adata_subset.obs[coord_col].values

        order = np.argsort(coords)
        coords_sorted = coords[order]

        bins = np.linspace(
            coords_sorted.min(),
            coords_sorted.max(),
            config_n_bins + 1
        )

        bin_indices = (
            np.digitize(coords_sorted, bins) - 1
        )

        bin_indices = np.clip(
            bin_indices,
            0,
            config_n_bins - 1
        )

        bin_centers = (
            bins[:-1] + bins[1:]
        ) / 2

        # ============================================================
        # 4. Calculate expression curves for each gene
        # ============================================================
        for gene in genes:

            if gene not in adata_subset.var_names:
                print(
                    f"Warning: {gene} was not found and will be skipped."
                )
                continue

            expr = adata_subset[:, gene].X

            if hasattr(expr, 'toarray'):
                expr = expr.toarray().ravel()
            else:
                expr = expr.ravel()

            expr_sorted = expr[order]

            bin_means = np.array([
                expr_sorted[bin_indices == i].mean()
                if np.any(bin_indices == i)
                else 0
                for i in range(config_n_bins)
            ])

            if normalization == 'minmax':

                bin_means_norm = (
                    bin_means - bin_means.min()
                ) / (
                    bin_means.max()
                    - bin_means.min()
                    + 1e-10
                )

            elif normalization == 'zscore':

                bin_means_norm = (
                    bin_means - bin_means.mean()
                ) / (
                    bin_means.std() + 1e-10
                )

            elif normalization == 'none':

                bin_means_norm = bin_means

            else:

                raise ValueError(
                    f"Unknown normalization method: {normalization}"
                )

            all_gene_curves[gene] = bin_means_norm

            # ============================================================
            # 5. Plot curves
            # ============================================================
            if config_color:

                color = config_color

            elif gene_colors and gene in gene_colors:

                color = gene_colors[gene]

            else:

                color = None

            if config_linestyle:

                linestyle = config_linestyle

            elif gene_linestyles and gene in gene_linestyles:

                linestyle = gene_linestyles[gene]

            else:

                linestyle = '-'

            if smooth > 0:

                x_smooth = np.linspace(
                    bin_centers.min(),
                    bin_centers.max(),
                    config_n_bins * 10
                )

                try:

                    k = min(
                        int(smooth),
                        len(bin_centers) - 1,
                        5
                    )

                    spl = make_interp_spline(
                        bin_centers,
                        bin_means_norm,
                        k=k
                    )

                    y_smooth = spl(x_smooth)

                    if normalization == 'minmax':
                        y_smooth = np.clip(
                            y_smooth,
                            0,
                            1
                        )

                    if fill:

                        current_ax.fill_between(
                            x_smooth,
                            y_smooth,
                            alpha=fill_alpha,
                            color=color
                        )

                    current_ax.plot(
                        x_smooth,
                        y_smooth,
                        linestyle=linestyle,
                        label=gene,
                        linewidth=linewidth,
                        color=color,
                        alpha=0.7
                    )

                    if show_points:

                        current_ax.plot(
                            bin_centers,
                            bin_means_norm,
                            'o',
                            markersize=4,
                            alpha=0.5,
                            color=color
                        )

                except Exception as e:

                    print(
                        f"Warning: smoothing failed for {gene} "
                        f"({str(e)}); using the original curve."
                    )

                    marker = (
                        'o-' if show_points else '-'
                    )

                    if fill:

                        current_ax.fill_between(
                            bin_centers,
                            bin_means_norm,
                            alpha=fill_alpha,
                            color=color
                        )

                    current_ax.plot(
                        bin_centers,
                        bin_means_norm,
                        marker,
                        label=gene,
                        linewidth=linewidth,
                        color=color,
                        linestyle=(
                            linestyle
                            if not show_points
                            else '-'
                        ),
                        markersize=(
                            4 if show_points else 0
                        ),
                        alpha=0.7
                    )

            else:

                marker = (
                    'o-' if show_points else '-'
                )

                if fill:

                    current_ax.fill_between(
                        bin_centers,
                        bin_means_norm,
                        alpha=fill_alpha,
                        color=color
                    )

                current_ax.plot(
                    bin_centers,
                    bin_means_norm,
                    marker,
                    label=gene,
                    linewidth=linewidth,
                    color=color,
                    linestyle=(
                        linestyle
                        if not show_points
                        else '-'
                    ),
                    markersize=(
                        4 if show_points else 0
                    ),
                    alpha=0.7
                )

    # ============================================================
    # 6. Add vertical reference lines
    # ============================================================
    if vertical_lines:

        for line_pos in vertical_lines:

            ax.axvline(
                x=line_pos,
                color=vline_color,
                linestyle=vline_style,
                linewidth=vline_width,
                alpha=vline_alpha,
                zorder=10
            )

    # ============================================================
    # 7. Configure axes
    # ============================================================
    if xlim:
        ax.set_xlim(xlim)

    if ylim:
        ax.set_ylim(ylim)

    if secondary_ylim and ax2:
        ax2.set_ylim(secondary_ylim)

    ax.set_xlabel(
        f'{coord_col} Position',
        fontsize=12
    )

    ylabel_dict = {
        'minmax': 'Normalized Expression (0-1)',
        'zscore': 'Z-score Normalized Expression',
        'none': 'Expression Level'
    }

    ax.set_ylabel(
        ylabel_dict.get(
            normalization,
            'Expression'
        ),
        fontsize=12
    )

    if ax2:

        ax2.set_ylabel(
            f'{ylabel_dict.get(normalization, "Expression")} '
            f'(Secondary)',
            fontsize=12
        )

    ax.set_title(
        f'Gene Expression along {coord_col} axis',
        fontsize=14,
        fontweight='bold'
    )

    ax.xaxis.set_major_locator(
        mticker.MultipleLocator(
            x_tick_interval
        )
    )

    # ============================================================
    # 8. Legend
    # ============================================================
    if show_legend:

        lines1, labels1 = (
            ax.get_legend_handles_labels()
        )

        if ax2:

            lines2, labels2 = (
                ax2.get_legend_handles_labels()
            )

            ax.legend(
                lines1 + lines2,
                labels1 + labels2,
                bbox_to_anchor=(1.15, 1),
                loc='upper left'
            )

        else:

            ax.legend(
                bbox_to_anchor=(1.05, 1),
                loc='upper left'
            )

    ax.margins(
        x=0,
        y=0.02
    )

    plt.tight_layout()

    return fig, all_gene_curves

In [ ]:
adata_w22 = fetal[fetal.obs['roi'] == '1_W22_22'].copy()

In [ ]:
vertical_lines = [-150]
fig = plot_binned_density(
    adata_w22, 
    n_bins=40,
    palette=custom_palette,
    figsize=(35, 12),
    smooth=2,  # 轻微平滑
    show_legend=True,
    linewidth=6,
    vertical_lines=vertical_lines,
    #vline_width=6,
    density_method='count',
    coord_col='DV',
    group_col='leiden_germ',
    #secondary_y_groups=["8"], 
    #dashed_groups=["8"],
    #secondary_ylim=(0, 300),
    #x_tick_interval=50,
    fill_alpha=0.5,
    #xlim=(-100,270),
    #ylim=(0,120),
   primary_y_groups=['5', '6',"10","8","7","4","3","2"]
)
#plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\2st_2025_10_07\Figure3\density_plot_midgut_AP.pdf', dpi=900, bbox_inches='tight')
plt.show()

## 3.7 Fig2.j

In [ ]:
adata_sub = fetal[fetal.obs["leiden_c5"].astype(str).isin(["6","2","5.0"])].copy()
# 检查筛选后是否为空
if adata_sub.n_obs == 0:
    print("Warning: adata_sub is empty. Aborting processing.")
else:
    print(f"Info: adata_sub selected with {adata_sub.n_obs} cells.")
    
    if 'counts_raw' in adata_sub.layers:
        adata_sub.X = adata_sub.layers['counts_raw'].copy()
        print("Info: .X initialized from 'counts_raw' layer.")
    
    sc.pp.normalize_total(adata_sub, inplace=True)
    sc.pp.log1p(adata_sub)
    sc.pp.highly_variable_genes(adata_sub, n_top_genes=2000)
    adata_sub.raw = adata_sub.copy()
    sc.pp.scale(adata_sub, max_value=10)
    sc.tl.pca(adata_sub, use_highly_variable=True)
    sc.pp.neighbors(adata_sub)
    sc.tl.umap(adata_sub, min_dist=0.01, spread=2)
    sc.tl.leiden(adata_sub, resolution=0.7, key_added='leiden_0.7')
    
    print(f"adata_sub (n={adata_sub.n_obs}) processing complete.")
    print(f"Total genes retained: {adata_sub.n_vars}")
    print(f"Highly variable genes used for analysis: {adata_sub.var.highly_variable.sum()}")

In [ ]:

adata_sub.obs["leiden_final"] = adata_sub.obs["leiden_0.7"].astype(str)


adata_sub.obs["leiden_final"] = adata_sub.obs["leiden_final"].replace({"6": "1", "5": "0"})


adata_sub.obs["leiden_final"] = (
    adata_sub.obs["leiden_final"].astype("category")
    .cat.remove_unused_categories()
)

print(adata_sub.obs["leiden_final"].value_counts())

In [ ]:
custom_palette={
    "0": "#9d00ff",  
    "1": "#0033cc",  
    "2": "#ff1493", 
    "3": "#ffee00",  
    "4": "#00e0b8",  
}

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

umap_coords = adata_sub.obsm["X_umap"]

leiden_clusters = adata_sub.obs["leiden_final"].astype(str)

umap_df = pd.DataFrame(
    umap_coords,
    columns=["UMAP_1", "UMAP_2"],
    index=adata_sub.obs_names
)

umap_df["leiden_final"] = leiden_clusters.values

print("Clusters in leiden_final:")
print(umap_df["leiden_final"].value_counts().sort_index())

print("Palette keys:")
print(custom_palette.keys())

plt.figure(figsize=(6, 4))

sns.scatterplot(
    data=umap_df,
    x="UMAP_1",
    y="UMAP_2",
    hue="leiden_final",
    palette=custom_palette,
    s=3,
    edgecolor=None,
    linewidth=0
)

plt.title("UMAP Plot with Leiden Final Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")

plt.legend(
    title="Leiden final",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    markerscale=5,
    frameon=False
)

plt.tight_layout()

plt.savefig(
    r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\umap\umap_pregranulosa_cell_celline_res0.7.png",
    format="png",
    dpi=900,
    bbox_inches="tight"
)

plt.show()

## 3.8 Fig.2k

In [ ]:
gene_list = ["FOXL2",'EPCAM', 'IGF2BP3', 'JAG1', 'SDC2', 'COL4A2', 'MRC2', 'PEG10', 'MEG3',
         'NPNT', 'KITLG', 'LDHA', 'PKM', 'MCM4', 'SMC3', 'MYBL2']


genes_present = [g for g in gene_list if g in adata.var_names]
genes_absent  = [g for g in gene_list if g not in adata.var_names]


sc.pl.dotplot(
    adata_sub,
    var_names=genes_present,
    groupby='leiden_final',              
    standard_scale='var',         
    cmap='Reds',
    dendrogram=True,              
    figsize=(5.5,2),
    save='granulosa_cell_heatmap.pdf'
)

In [ ]:
adata_sub.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\adata\5st_12st_Figure2_5st\pregranulosa_cellline_8st.h5ad")

## 3.9 Fig.2m

In [ ]:
import pandas as pd

df9 = pd.read_csv(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\cellnest_adata\1st_2026_03_26\W9\output\MYDATA_3D\CellNEST_MYDATA_3D_top20percent.csv")

df16 = pd.read_csv(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\cellnest_adata\1st_2026_03_26\W16\output\MYDATA_3D\CellNEST_MYDATA_3D_top20percent.csv")

df22 = pd.read_csv(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\cellnest_adata\1st_2026_03_26\W22\output\MYDATA_3D\CellNEST_MYDATA_3D_top20percent.csv")

In [ ]:
df_all = pd.concat([
    df9.assign(area='9'),
   
    df16.assign(area='16'),
  
    df22.assign(area='22')
], ignore_index=True)

In [ ]:
df=df_all.copy()
fetal.obs['cell_id'] = fetal.obs.index

In [ ]:

m = fetal.obs.set_index('cell_id')['leiden_germ']


df['leiden_from'] = df['from_cell'].map(m)


df['leiden_to'] = df['to_cell'].map(m)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

selected_pairs = [
    ('gm0', 'gc1'),
    ('gm3', 'gc2'),
    ('gm4', 'gc2'),
    ('gm7', 'gc2'),
    ('gm5', 'gc1'),
    ('gm6', 'gc4'),
    ('gm10', 'gc3'),
    ('gm8', 'gc0')
]

real_niche_counts = {
    'other': 123999,
    'gc0': 16610,
    'gc1': 12866,
    'gc2': 9919,
    'gc3': 7121,
    'gc4': 6111,
    'gm0': 5141,
    'gm1': 4151,
    'gm2': 3971,
    'gm3': 3661,
    'gm4': 3650,
    'gm5': 2766,
    'gm6': 2174,
    'gm7': 2095,
    'gm8': 1917,
    'gm9': 1071,
    'gm10': 274,
}

df['leiden_from'] = df['leiden_from'].astype(str)
df['leiden_to'] = df['leiden_to'].astype(str)
df['lr_pair'] = df['ligand'] + '-' + df['receptor']

niche_pairs = [
    f"{f}→{t}"
    for f, t in selected_pairs
    if len(
        df[
            (df['leiden_from'] == f) &
            (df['leiden_to'] == t)
        ]
    ) > 0
]

top5_lr_per_pair = {}

for from_niche, to_niche in selected_pairs:
    np_name = f"{from_niche}→{to_niche}"

    if np_name not in niche_pairs:
        continue

    sub = df[
        (df["leiden_from"] == from_niche) &
        (df["leiden_to"] == to_niche)
    ]

    top5 = (
        sub.groupby("lr_pair")["attention_score"]
        .count()
        .nlargest(5)
        .index
        .tolist()
    )

    top5_lr_per_pair[np_name] = top5

all_lr_to_show = []

for np_name in niche_pairs:
    for lr in top5_lr_per_pair.get(np_name, []):
        if lr not in all_lr_to_show:
            all_lr_to_show.append(lr)

print(f"{len(all_lr_to_show)} LR pairs before filtering")
print(all_lr_to_show)

excluded_lr_pairs = {
    "GRN-CLEC4M",
    "JAG1-CD46",
    "GDF9-KIT",
    "APP-SORL1",
    "APP-NCSTN",
    "NPTN-DLG4",
    "CLSTN1-IGF2R",
    "CLSTN1-CD47",
    "CLSTN1-TFRC",
}

all_lr_to_show = [
    lr
    for lr in all_lr_to_show
    if lr not in excluded_lr_pairs
]

print(f"{len(all_lr_to_show)} LR pairs after filtering")
print(all_lr_to_show)

results = []

for from_niche, to_niche in selected_pairs:
    np_name = f"{from_niche}→{to_niche}"

    if np_name not in niche_pairs:
        continue

    sub = df[
        (df['leiden_from'] == from_niche) &
        (df['leiden_to'] == to_niche)
    ]

    sender_count = real_niche_counts.get(
        from_niche,
        1
    )

    for lr in all_lr_to_show:
        lr_sub = sub[
            sub['lr_pair'] == lr
        ]

        count = len(lr_sub)

        mean_score = (
            lr_sub['attention_score'].mean()
            if count > 0
            else np.nan
        )

        norm_count = count / sender_count

        results.append({
            'niche_pair': np_name,
            'lr_pair': lr,
            'count': count,
            'mean_score': mean_score,
            'norm_count': norm_count
        })

plot_df = pd.DataFrame(results)

lr_order = (
    plot_df.groupby('lr_pair')['count']
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

lr_idx = {
    lr: i
    for i, lr in enumerate(lr_order)
}

np_idx = {
    np_: i
    for i, np_ in enumerate(niche_pairs)
}

plot_df_nonzero = (
    plot_df[
        plot_df['count'] > 0
    ]
    .copy()
)

x = plot_df_nonzero['lr_pair'].map(lr_idx)
y = plot_df_nonzero['niche_pair'].map(np_idx)

size = (
    plot_df_nonzero['norm_count'] * 2000
)

color = plot_df_nonzero['mean_score']

fig, ax = plt.subplots(
    figsize=(5.5, 4.5)
)

color = plot_df_nonzero['norm_count']

sc = ax.scatter(
    y,
    x,
    s=size,
    color='#c0392b',
    alpha=0.9,
    vmin=0,
    vmax=0.2,
    edgecolors='none',
    linewidths=0.3
)

ax.set_yticks(
    range(len(lr_order))
)

ax.set_yticklabels(
    lr_order,
    fontsize=8
)

ax.set_xticks(
    range(len(niche_pairs))
)

ax.set_xticklabels(
    niche_pairs,
    rotation=90,
    ha='right',
    fontsize=10
)

ax.set_ylabel(
    'Ligand-Receptor Pair'
)

ax.set_xlabel(
    'Niche Pair (From → To)'
)

ax.grid(False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for s_val in [0.01, 0.05, 0.1, 0.2]:
    ax.scatter(
        [],
        [],
        s=s_val * 2000,
        c='gray',
        alpha=0.6,
        label=f'{s_val:.2f}'
    )

ax.legend(
    title='Norm. Count',
    bbox_to_anchor=(1.3, 1),
    loc='upper left',
    frameon=False,
    fontsize=8,
    title_fontsize=9
)

plt.tight_layout()

plt.savefig(
    r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\cellnest\LR_bubble_plot_germ.pdf',
    bbox_inches='tight'
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

selected_pairs = [
    ('gc1', 'gm0'),
    ('gc2', 'gm3'),
    ('gc2', 'gm4'),
    ('gc2', 'gm7'),
    ('gc1', 'gm5'),
    ('gc4', 'gm6'),
    ('gc3', 'gm10'),
    ('gc0', 'gm8')
]

real_niche_counts = {
    'other': 123999,
    'gc0': 16610,
    'gc1': 12866,
    'gc2': 9919,
    'gc3': 7121,
    'gc4': 6111,
    'gm0': 5141,
    'gm1': 4151,
    'gm2': 3971,
    'gm3': 3661,
    'gm4': 3650,
    'gm5': 2766,
    'gm6': 2174,
    'gm7': 2095,
    'gm8': 1917,
    'gm9': 1071,
    'gm10': 274,
}

df['leiden_from'] = df['leiden_from'].astype(str)
df['leiden_to'] = df['leiden_to'].astype(str)
df['lr_pair'] = df['ligand'] + '-' + df['receptor']

niche_pairs = [
    f"{f}→{t}"
    for f, t in selected_pairs
    if len(
        df[
            (df['leiden_from'] == f) &
            (df['leiden_to'] == t)
        ]
    ) > 0
]

top5_lr_per_pair = {}

for from_niche, to_niche in selected_pairs:
    np_name = f"{from_niche}→{to_niche}"

    if np_name not in niche_pairs:
        continue

    sub = df[
        (df["leiden_from"] == from_niche) &
        (df["leiden_to"] == to_niche)
    ]

    top5 = (
        sub.groupby("lr_pair")["attention_score"]
        .count()
        .nlargest(5)
        .index
        .tolist()
    )

    top5_lr_per_pair[np_name] = top5

all_lr_to_show = []

for np_name in niche_pairs:
    for lr in top5_lr_per_pair.get(np_name, []):
        if lr not in all_lr_to_show:
            all_lr_to_show.append(lr)

print(f"{len(all_lr_to_show)} LR pairs before filtering")
print(all_lr_to_show)

excluded_lr_pairs = {
    "BMP2-FGFR3",
    "VEGFA-PIK3CB",
    "GMFB-ITPR3",
    "APP-NCSTN",
    "APP-SORL1",
    "PTPRF-INSR",
    "PTPRF-MET",
    "CLSTN1-TFRC",
    "CLSTN1-IGF2R",
    "CLSTN1-CD47",
    "PLXNB2-PTCH1",
    "GDF9-KIT",
    "JAG1-CD46",
    "BMP6-BMPR1A",
}

all_lr_to_show = [
    lr
    for lr in all_lr_to_show
    if lr not in excluded_lr_pairs
]

print(f"{len(all_lr_to_show)} LR pairs after filtering")
print(all_lr_to_show)

results = []

for from_niche, to_niche in selected_pairs:
    np_name = f"{from_niche}→{to_niche}"

    if np_name not in niche_pairs:
        continue

    sub = df[
        (df['leiden_from'] == from_niche) &
        (df['leiden_to'] == to_niche)
    ]

    sender_count = real_niche_counts.get(from_niche, 1)

    for lr in all_lr_to_show:
        lr_sub = sub[sub['lr_pair'] == lr]

        count = len(lr_sub)

        mean_score = (
            lr_sub['attention_score'].mean()
            if count > 0
            else np.nan
        )

        norm_count = count / sender_count

        results.append({
            'niche_pair': np_name,
            'lr_pair': lr,
            'count': count,
            'mean_score': mean_score,
            'norm_count': norm_count
        })

plot_df = pd.DataFrame(results)

lr_order = (
    plot_df.groupby('lr_pair')['count']
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

lr_idx = {
    lr: i
    for i, lr in enumerate(lr_order)
}

np_idx = {
    np_: i
    for i, np_ in enumerate(niche_pairs)
}

plot_df_nonzero = (
    plot_df[
        plot_df['count'] > 0
    ]
    .copy()
)

x = plot_df_nonzero['lr_pair'].map(lr_idx)
y = plot_df_nonzero['niche_pair'].map(np_idx)

size = plot_df_nonzero['norm_count'] * 2000
color = plot_df_nonzero['mean_score']

fig, ax = plt.subplots(figsize=(5.5, 4.5))

color = plot_df_nonzero['norm_count']

sc = ax.scatter(
    y,
    x,
    s=size,
    color='#457B9D',
    alpha=0.9,
    vmin=0,
    vmax=0.2,
    edgecolors='none',
    linewidths=0.3
)

ax.set_yticks(range(len(lr_order)))
ax.set_yticklabels(lr_order, fontsize=8)

ax.set_xticks(range(len(niche_pairs)))
ax.set_xticklabels(
    niche_pairs,
    rotation=90,
    ha='right',
    fontsize=10
)

ax.set_ylabel('Ligand-Receptor Pair')
ax.set_xlabel('Niche Pair (From → To)')

ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for s_val in [0.01, 0.05, 0.1, 0.2]:
    ax.scatter(
        [],
        [],
        s=s_val * 2000,
        c='gray',
        alpha=0.6,
        label=f'{s_val:.2f}'
    )

ax.legend(
    title='Norm. Count',
    bbox_to_anchor=(1.3, 1),
    loc='upper left',
    frameon=False,
    fontsize=8,
    title_fontsize=9
)

plt.tight_layout()

plt.savefig(
    r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\3st_2026_07_16\Figure2\cellnest\LR_bubble_plot_pregranulosa.pdf',
    bbox_inches='tight'
)

plt.show()

# 4. Fig.3

## 4.1 Fig.3a

In [ ]:
adult_o = postnatal[postnatal.obs['leiden_0.55'] == '13'].copy()
fetal_o = fetal[fetal.obs['leiden_c5'] == '5.2'].copy()

In [ ]:
import anndata as ad

common_cols = list(set(fetal_o.obs.columns) & set(adult_o.obs.columns))
print("Common columns:", common_cols)

fetal_o_slim = fetal_o.copy()
fetal_o_slim.obs = fetal_o_slim.obs[common_cols]

adult_o_slim = adult_o.copy()
adult_o_slim.obs = adult_o_slim.obs[common_cols]

common_genes = list(set(fetal_o.var_names) & set(adult_o.var_names))
print(f"Number of common genes: {len(common_genes)}")

fetal_o_slim = fetal_o_slim[:, common_genes]
adult_o_slim = adult_o_slim[:, common_genes]

adata_combined = ad.concat(
    [fetal_o_slim, adult_o_slim],
    join="inner",
    label="source",
    keys=["fetal", "adult"]
)

print(adata_combined)

In [ ]:
adata_combined.layers['counts_raw']=adata_combined.X.copy()

In [ ]:
adata_sub = adata_combined.copy()

sc.pp.normalize_total(adata_sub)
sc.pp.log1p(adata_sub)

sc.pp.highly_variable_genes(
    adata_sub,
    n_top_genes=2000,
    batch_key="batch",
    subset=False
)

print(f"Number of HVGs: {adata_sub.var['highly_variable'].sum()}")

sc.pp.scale(adata_sub)

sc.tl.pca(
    adata_sub,
    n_comps=15,
    use_highly_variable=True
)

sc.pp.neighbors(
    adata_sub,
    use_rep="X_pca",
    n_neighbors=15,
    random_state=42
)

sc.tl.umap(
    adata_sub,
    random_state=42
)

sc.tl.leiden(
    adata_sub,
    resolution=0.6,
    key_added="leiden_sub_0.6"
)

sc.pl.umap(
    adata_sub,
    color=["leiden_sub_0.6"],
    size=50
)

In [ ]:
custom_palette = {
    "0": "#E83232",
    "1": "#3A8FD9",
    "2": "#8B45C8",
    "3": "#F07A20",
    "4": "#00BFFF",
    "5": "#D95F9E",
    "6": "#1AAFAF",
    "7": "#C9A020",
    "8": "#3DBD3D",
}

In [ ]:

tsne_coords =adata.obsm['X_umap']

leiden_clusters = adata.obs['leiden_sub_0.6']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['tSNE_1', 'tSNE_2'])
tsne_df['leiden_sub_0.6'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3.5))
sns.scatterplot(data=tsne_df, x='tSNE_1', y='tSNE_2', hue='leiden_sub_0.6', palette=custom_palette, s=10, edgecolor=None)
plt.title("t-SNE Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 3\umap\umap_oocyte_cluster.png', format="png",dpi=900,bbox_inches='tight')
plt.show()


In [ ]:
adata_sub.write(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adult_ovary_oocyte_res0.6.h5ad")

## 4.2 Fig.3b

In [ ]:
adata=sc.read_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adult_ovary_oocyte_res0.6.h5ad")

In [ ]:
my_genes=[
"BMP6","KIT","LGR4","SFRP1",'GRM5','PDIA2',"DUOX2","CASP7","AXL","RPRM", "KHDC3L","DAG1","CGB3","LEP","GSTM3","ALDOA","PIEZO1","FCGBP",
]
sc.pl.dotplot(
    adata,
    var_names=my_genes,
    groupby='leiden_sub_0.6',  
    standard_scale='var',  
    #vmax=2, 
    cmap="Reds",
    figsize=(7, 3.2),
    dendrogram=True,
    save='OOcyte_leiden_sub0.6_normalize_2ST.pdf'
)

## 4.3 Fig.3c

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

CLUSTER_KEY = 'leiden_sub_0.6'
AGE_KEY = 'age'
ROI_KEY = 'roi'

custom_palette = {
    0: "#E83232",
    1: "#3A8FD9",
    2: "#8B45C8",
    3: "#F07A20",
    4: "#00BFFF",
    5: "#D95F9E",
    6: "#1AAFAF",
    7: "#C9A020",
    8: "#3DBD3D",
}

cluster_colors = {
    str(k): v
    for k, v in custom_palette.items()
}

cluster_labels = {
    '4': 'FeO',
    '0': 'pPdO-S',
    '1': 'pPdO-M',
    '3': 'aPdO',
    '6': 'AcO',
    '2': 'PrO',
    '7': 'dGcO',
    '5': 'mSO',
    '8': 'ScO',
}

def age_to_float(age):
    try:
        return float(age)
    except:
        return np.nan

adata.obs['age_numeric'] = adata.obs[AGE_KEY].apply(age_to_float)

oocyte_clusters = [
    '0', '1', '2', '3', '4',
    '5', '6', '7', '8'
]

adata_oo = adata[
    adata.obs[CLUSTER_KEY].isin(oocyte_clusters)
].copy()

adata_oo.obs['age_numeric'] = (
    adata_oo.obs[AGE_KEY]
    .apply(age_to_float)
)

TRANS_ROI = '4_FTM117_28'

adata_oo.obs['is_trans_28'] = (
    adata_oo.obs[ROI_KEY] == TRANS_ROI
)

def assign_age_group(row):
    age = row['age_numeric']
    is_trans = row['is_trans_28']

    if is_trans:
        return None

    if pd.isna(age):
        return None

    if age < 1:
        return 'Fetal'
    elif age <= 4:
        return '4 yr'
    elif age <= 12:
        return '7-12 yr'
    elif age <= 18:
        return '13-18 yr'
    else:
        return '>18 yr'

adata_oo.obs['age_group'] = (
    adata_oo.obs
    .apply(assign_age_group, axis=1)
)

adata_oo_regular = adata_oo[
    adata_oo.obs['age_group'].notna()
].copy()

adata_oo_trans = adata_oo[
    adata_oo.obs['is_trans_28']
].copy()

age_group_order = [
    'Fetal',
    '4 yr',
    '7-12 yr',
    '13-18 yr',
    '>18 yr'
]

cluster_order = [
    '8', '7', '6', '5', '4',
    '3', '2', '1', '0'
]

bar_width = 0.92

ct_regular = pd.crosstab(
    adata_oo_regular.obs['age_group'],
    adata_oo_regular.obs[CLUSTER_KEY],
    normalize='index',
) * 100

ct_regular = ct_regular.reindex(
    columns=[
        c for c in cluster_order
        if c in ct_regular.columns
    ],
    fill_value=0
)

ct_regular = ct_regular.reindex(
    index=[
        g for g in age_group_order
        if g in ct_regular.index
    ]
)

if len(adata_oo_trans) > 0:

    trans_counts = (
        adata_oo_trans.obs[CLUSTER_KEY]
        .value_counts()
    )

    trans_pct = pd.Series(
        0.0,
        index=cluster_order
    )

    for c in cluster_order:
        if c in trans_counts.index:
            trans_pct[c] = (
                trans_counts[c]
                / len(adata_oo_trans)
                * 100
            )

else:

    trans_pct = pd.Series(
        0.0,
        index=cluster_order
    )

ct_trans_df = pd.DataFrame(
    trans_pct.values.reshape(1, -1),
    columns=cluster_order,
    index=['Trans* (28 yr)']
)

ct_full = pd.concat([
    ct_regular,
    ct_trans_df
])

n_regular = (
    adata_oo_regular.obs
    .groupby('age_group')
    .size()
)

n_regular = n_regular.reindex([
    g for g in age_group_order
    if g in n_regular.index
])

n_trans = len(adata_oo_trans)

n_all = dict(n_regular)
n_all['Trans* (28 yr)'] = n_trans

print("=" * 50)
print("Cells per group:")

for grp in ct_full.index:
    print(
        f"  {grp}: "
        f"{n_all.get(grp, 0)}"
    )

print()

print("Cluster proportions:")
print(
    ct_full
    .round(1)
    .to_string()
)

n_bars = len(ct_full)

fig, ax = plt.subplots(
    figsize=(5, 3)
)

x = np.arange(n_bars)
bar_width = 0.85
bottom = np.zeros(n_bars)

for cluster_id in cluster_order:

    if cluster_id not in ct_full.columns:
        continue

    vals = ct_full[cluster_id].values

    color = cluster_colors.get(
        cluster_id,
        'gray'
    )

    label = (
        f'C{cluster_id} '
        f'({cluster_labels.get(cluster_id, cluster_id)})'
    )

    ax.bar(
        x,
        vals,
        bar_width,
        bottom=bottom,
        color=color,
        label=label,
        edgecolor='white',
        linewidth=0.4,
    )

    bottom += vals

xlabels = [
    f'{grp}\n(n={n_all.get(grp, 0)})'
    for grp in ct_full.index
]

ax.set_xticks(x)

ax.set_xticklabels(
    xlabels,
    fontsize=9
)

ax.axvline(
    x=n_bars - 1.5,
    color='black',
    linestyle='--',
    linewidth=1,
    alpha=0.6
)

ax.text(
    n_bars - 1,
    102,
    'Transgender\n(testosterone)',
    ha='center',
    va='bottom',
    fontsize=8,
    color='#555555',
    style='italic',
)

ax.set_ylabel(
    'Proportion (%)',
    fontsize=11
)

ax.set_ylim(
    0,
    108
)

ax.set_xlim(
    -0.5,
    n_bars - 0.3
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_handles = [
    mpatches.Patch(
        color=cluster_colors[c],
        label=f'C{c} ({cluster_labels[c]})'
    )
    for c in cluster_order
    if c in cluster_colors
]

ax.legend(
    handles=legend_handles,
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    fontsize=8.5,
    frameon=True,
    framealpha=0.9,
    edgecolor='lightgray',
)

plt.tight_layout()

plt.savefig(
    r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 3\barplot\barplot_oocyte_cluster_age_v3.pdf',
    format="pdf",
    bbox_inches='tight'
)

plt.show()

print("Saved: barplot_oocyte_cluster_age_v3.pdf")

## 4.4 Fig.3f

In [ ]:
sc.pl.umap(adata_sub, color=['CGB3', 'LEP', 'LEPR', 'PLIN4', 'STAB1', 'GCKR'], size=250,ncols=2,show=False)
plt.savefig(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 3\umap\gene\genes_oocyte.pdf", bbox_inches='tight')

## 4.4 Fig.3g

In [ ]:
adata_sub = adata[adata.obs['leiden_sub_0.6'].isin(["2",'3','8'])].copy()
tsne_coords =adata_sub.obsm['X_umap']

leiden_clusters = adata_sub.obs['leiden_sub_0.6']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['tSNE_1', 'tSNE_2'])
tsne_df['leiden_sub_0.6'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3.5))
sns.scatterplot(data=tsne_df, x='tSNE_1', y='tSNE_2', hue='leiden_sub_0.6', palette=custom_palette, s=10, edgecolor=None)
plt.title("t-SNE Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 3\umap\umap_oocyte_adult_cluster.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

## 4.4 Fig.3h

In [ ]:
sc.pl.umap(adata_sub, color=["CDKN1B","GDF9"], size=250,ncols=2,show=False)
plt.savefig(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 3\umap\gene\genes_CDKN1B_GDF9.pdf", bbox_inches='tight')

# 5.Fig.4

In [ ]:
import scanpy as sc


file_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adulit_ovary_raw_with_cluster.h5ad"

print("正在读取文件...")
ovary = sc.read_h5ad(file_path)

In [ ]:
gc= sc.read_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adult_ovary_without_OVA177_GC_RES0.55.h5ad")
TC= sc.read_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adult_ovary_without_OVA177_itc_RES0.45.h5ad")
bl= sc.read_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adult_ovary_without_OVA177_bl_RES0.3.h5ad")
im= sc.read_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adult_ovary_without_OVA177_im_RES0.5.h5ad")

In [ ]:
import pandas as pd

ovary.obs['merge'] = 'other'
ovary.obs['merge'] = ovary.obs['merge'].astype('object')

common_cells = ovary.obs.index.intersection(im.obs.index)

im_values = (
    'im' +
    im.obs.loc[
        common_cells,
        'leiden_sub_0.5'
    ].astype(str).values
)

ovary.obs.loc[
    common_cells,
    'merge'
] = im_values

ovary.obs['merge'] = (
    ovary.obs['merge']
    .astype('category')
)

print(
    ovary.obs['merge']
    .value_counts()
)




import pandas as pd


ovary.obs['merge'] = ovary.obs['merge'].astype('object')


common_cells = ovary.obs.index.intersection(bl.obs.index)


bl_values = (
    'bl' +
    bl.obs.loc[
        common_cells,
        'leiden_sub_0.3'
    ].astype(str).values
)


ovary.obs.loc[
    common_cells,
    'merge'
] = bl_values


ovary.obs['merge'] = (
    ovary.obs['merge']
    .astype('category')
)


print(
    ovary.obs['merge']
    .value_counts()
)

import pandas as pd

ovary.obs['merge'] = ovary.obs['merge'].astype('object')

common_cells = ovary.obs.index.intersection(adata.obs.index)

gc_values = (
    'gc' +
    adata.obs.loc[
        common_cells,
        'leiden_sub_0.55'
    ].astype(str).values
)

ovary.obs.loc[
    common_cells,
    'merge'
] = gc_values

ovary.obs['merge'] = (
    ovary.obs['merge']
    .astype('category')
)

print(
    ovary.obs['merge']
    .value_counts()
)


import pandas as pd

ovary.obs['merge'] = ovary.obs['merge'].astype('object')

common_cells = ovary.obs.index.intersection(oocyte.obs.index)

oo_values = (
    'oo' +
    oocyte.obs.loc[
        common_cells,
        'leiden_sub_0.6'
    ].astype(str).values
)

ovary.obs.loc[
    common_cells,
    'merge'
] = oo_values

ovary.obs['merge'] = (
    ovary.obs['merge']
    .astype('category')
)

print(
    ovary.obs['merge']
    .value_counts()
)


ovary.obs['merge'] = ovary.obs['merge'].astype('object')

mask = ovary.obs['merge'] == 'other'
ovary.obs.loc[mask, 'merge'] = 'ov' + ovary.obs.loc[mask, 'leiden_1'].astype(str)

ovary.obs['merge'] = ovary.obs['merge'].astype('category')
print(ovary.obs['merge'].value_counts())

In [ ]:
import pandas as pd
import os

folder = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 4\from_xenium\cell_type1"

ovary.obs['follicle_type'] = 'other'


for fname in os.listdir(folder):
    if not fname.endswith('.csv'):
        continue
    label = os.path.splitext(fname)[0]  
    df = pd.read_csv(os.path.join(folder, fname), skiprows=2)
    matched = ovary.obs['cell_id'].isin(df['Cell ID'])
    ovary.obs.loc[matched, 'follicle_type'] = label
    print(f"{label}: {matched.sum()} cells matched")

print(ovary.obs['follicle_type'].value_counts())

In [ ]:
import numpy as np

batch_offset = {
    '29201': 0,
    '29208': 12000,
    '41720': 24000,
    '53080': 36000,
    '53256': 48000
}

ovary.obs['x_centroid_new'] = ovary.obs['x_centroid'].copy()
ovary.obs['y_centroid_new'] = ovary.obs['y_centroid'].copy()

for batch, offset in batch_offset.items():
    mask = ovary.obs['batch'] == batch
    ovary.obs.loc[mask, 'x_centroid_new'] = ovary.obs.loc[mask, 'x_centroid'] + offset

for batch in ovary.obs['batch'].unique():
    batch_obs = ovary.obs[ovary.obs['batch'] == batch]
    x_min = batch_obs['x_centroid_new'].min()
    x_max = batch_obs['x_centroid_new'].max()
    print(f"batch {batch}: x=[{x_min:.1f}, {x_max:.1f}]")

In [ ]:
from scipy.spatial import cKDTree


ovary.obs['follicle'] = 'other'


coords = ovary.obs[['x_centroid_new', 'y_centroid_new']].values
tree = cKDTree(coords)


oo_labels = {
    'oo0': 20,
    'oo1': 20,
    'oo3': 20,
    'oo2': 30,
    'oo5': 30,
}

for label, k in oo_labels.items():
    seed_idx = np.where(ovary.obs['merge'] == label)[0]
    if len(seed_idx) == 0:
        print(f"{label}: no cells found, skipped")
        continue
    _, neighbor_idx = tree.query(coords[seed_idx], k=k)
    all_idx = np.unique(neighbor_idx.flatten())
    ovary.obs.iloc[all_idx, ovary.obs.columns.get_loc('follicle')] = label

print(ovary.obs['follicle'].value_counts())

In [ ]:
ovary.obs['follicle_type_new'] = ovary.obs['follicle_type'].str.split('_').str[0]
print(ovary.obs['follicle_type_new'].value_counts())

ovary.obs['follicle_type_new'] = ovary.obs['follicle_type'].str.split('_').str[0]
print(ovary.obs['follicle_type_new'].value_counts())

mask = ovary.obs['follicle_type_new'] == 'other'
ovary.obs.loc[mask, 'follicle_type_new'] = ovary.obs.loc[mask, 'follicle']
print(ovary.obs['follicle_type_new'].value_counts())

In [ ]:
ovary.obs['follicle_type'] = ovary.obs['follicle_type'].astype('object')
ovary.obs['follicle_type'] = ovary.obs['follicle_type'].replace({'t3_3': 'st3_1', 't3_4': 'st3_2','t3_6': 't2_7' })
ovary.obs['follicle_type'] = ovary.obs['follicle_type'].astype('category')
print(ovary.obs['follicle_type'].value_counts())

subset = ovary[ovary.obs['follicle_type_new'] != 'other'].copy()
print(subset)


subset.obs['follicle_type_new'] = subset.obs['follicle_type_new'].astype('object')
subset.obs['follicle_type_new'] = subset.obs['follicle_type_new'].replace({'oo3': 'primordial', 'oo2': 'primary'})
subset.obs['follicle_type_new'] = subset.obs['follicle_type_new'].astype('category')
print(subset.obs['follicle_type_new'].value_counts())


follicle = subset[subset.obs['follicle_type_new'].isin(['primordial', 'primary', 'sf','ea', 'h', 'ba'])].copy()

## 5.1 Fig.4a

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import re
import pandas as pd


gc_colors = {f'gc{i}': c for i, c in enumerate([
    '#E6194B','#3CB44B','#FFE119','#4363D8','#F032E6','#911EB4',
    '#42D4F4','#F58231','#BFEF45','#FABEBE','#469990','#E6BEFF','#9A6324'
])}
tc_colors = {f'tc{i}': c for i, c in enumerate([
    '#00FFFF','#800000','#AAFFC3','#808000','#FFDAC1','#000075',
    '#A9A9A9','#DC143C','#FFFAC8','#FF1493','#7FFFD4'
])}
bl_colors = {f'bl{i}': c for i, c in enumerate([
    '#FF4500','#32CD32','#1E90FF','#FFD700','#8A2BE2','#00FA9A',
    '#FF69B4','#00CED1','#FF8C00','#9932CC'
])}
im_colors = {f'im{i}': c for i, c in enumerate([
    '#7FFF00','#4682B4','#B22222','#BA55D3','#6495ED','#DAA520',
    '#C71585','#20B2AA','#F4A460','#40E0D0','#FF6347','#00FF7F','#2E8B57'
])}
color_dict = {**gc_colors, **tc_colors, **bl_colors, **im_colors}

def sort_key(s):
    match = re.match(r'([a-zA-Z]+)(\d+)', s)
    if match:
        return (match.group(1), int(match.group(2)))
    return (s, 0)

def plot_follicle_composition(sub_adata, order=None, title=None, save_path=None):
    sub = sub_adata.obs
    if order is None:
        order = ['primordial', 'primary', 'sf', 'ea', 'h', 'ba']

 
    data, totals = {}, {}
    for ft in order:
        group = sub[sub['follicle_type_new'] == ft]['merge']
        totals[ft] = len(group)
        data[ft] = group.value_counts(normalize=True)

  
    df_plot = pd.DataFrame(data).fillna(0).T.reindex(order)

 
    all_merge = sorted(df_plot.columns.tolist(), key=sort_key)
    df_plot = df_plot[all_merge]

    
    fig, ax = plt.subplots(figsize=(5, 4.7))
    bottom = np.zeros(len(order))

    for col in reversed(all_merge):
        vals = df_plot[col].values
        color = color_dict.get(col, '#CCCCCC')
        ax.bar(order, vals, bottom=bottom, label=col, color=color,width=0.95)
        bottom += vals

    
    for i, ft in enumerate(order):
        ax.text(i, 1.01, f'n={totals[ft]}', ha='center', va='bottom', fontsize=9)

    ax.set_ylabel('Proportion')
    ax.set_xlabel('follicle_type_new')
    ax.set_ylim(0, 1.1)
    if title:
        ax.set_title(title)
    
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()


sub = follicle[follicle.obs['leiden_1'] == '3']
plot_follicle_composition(sub,save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 4\quantification\gc\grow_follicle_gc.pdf')

In [ ]:
ovary.write(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adulit_ovary_raw_with_cluster_follicle_type.h5ad")

In [ ]:
sub = follicle[follicle.obs['leiden_1'] == '1']
plot_follicle_composition(sub,save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 4\quantification\tc\grow_follicle_tc.pdf')

In [ ]:
sub = follicle[follicle.obs['leiden_1'] == '4']
plot_follicle_composition(sub,save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 4\quantification\bl\grow_follicle_bl.pdf')

In [ ]:
sub = follicle[follicle.obs['leiden_1'] == '0']
plot_follicle_composition(sub,save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 4\quantification\im\grow_follicle_im.pdf')

## 5.2 Fig.4d

In [ ]:
adata1 = gc[gc.obs['leiden_sub_0.55'].isin(["3","0","6"])].copy()

In [ ]:
my_genes=[
'CYP19A1', 'INHA', 'CYP11A1', 'FDX1', 'H19', 'HIF1A', 'CDH3', 'THBS1', 'GREM1', 'HTRA1','FST', 'NOTCH2']
sc.pl.dotplot(
    adata1,
    var_names=my_genes,
    groupby='leiden_sub_0.55',  
    standard_scale='var',  
    #vmax=2, 
    cmap="Reds",
    figsize=(6, 1),
    dendrogram=True,
    save='GC_cl036_leiden_sub0.55_normalize_2ST.pdf'
)

## 5.3 Fig.4g

In [ ]:
rest = sc.read_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\cellnest_adata\3st_2026_04_23_adult_ovary_follicletype\rest\MYDATA_3D_spatial\MYDATA_3D.h5ad")
rest.obs['merge'] = ovary.obs.loc[rest.obs.index, 'merge']



path2 = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\cellnest_adata\3st_2026_04_23_adult_ovary_follicletype\rest\1st\CellNEST_MYDATA_3D_top20percent.csv"
df = pd.read_csv(path2)

In [ ]:
adata_9 = ovary[ovary.obs['follicle_type_new'].isin(['ea', 'lt3','oo2', 'oo3','oo0', 'oo5','oo1', 'sf'])].copy()
print(adata_9)

In [ ]:

m = rest.obs.set_index('cell_id')['leiden_1']


df['leiden_from'] = df['from_cell'].map(m)


df['leiden_to'] = df['to_cell'].map(m)

In [ ]:
rest.obs['follicle_type_new'] = ovary.obs.loc[rest.obs.index, 'follicle_type_new']

In [ ]:

m = rest.obs.set_index('cell_id')['follicle_type_new']

df['follicle'] = df['from_cell'].map(m)

In [ ]:
def visualize_lr_network(df, from_leiden, to_leiden, top_n=15,
                          follicle_list=None,
                          cell_count_df=None,
                          bar_color='#4363D8',
                          figsize=None,
                          save_path=None):
 
    import matplotlib.pyplot as plt

    pair_df = df[(df['leiden_from'] == from_leiden) &
                 (df['leiden_to'] == to_leiden)].copy()

    if len(pair_df) == 0:
        print(f"❌ 没有找到数据")
        return

    def get_lr_summary(data, top_n, cell_count=None, follicle=None):
        summary = data.groupby(['ligand', 'receptor']).agg(
            avg_attention=('attention_score', 'mean'),
            count=('attention_score', 'count')
        ).reset_index()
        summary['pair'] = summary['ligand'] + '-' + summary['receptor']

        if cell_count is not None and follicle is not None and follicle != 'All':
            if follicle in cell_count.index and from_leiden in cell_count.columns:
                n_cells = cell_count.loc[follicle, from_leiden]
                summary['count_normalized'] = summary['count'] / n_cells * 100 if n_cells > 0 else summary['count']
            else:
                summary['count_normalized'] = summary['count']
        elif cell_count is not None and follicle == 'All':
            if from_leiden in cell_count.columns:
                n_cells_total = cell_count[from_leiden].sum()
                summary['count_normalized'] = summary['count'] / n_cells_total * 100
            else:
                summary['count_normalized'] = summary['count']
        else:
            summary['count_normalized'] = summary['count']

        plot_col = 'count_normalized' if cell_count is not None else 'count'
        summary = summary.nlargest(top_n, plot_col).sort_values(plot_col, ascending=True)
        return summary, plot_col

    def get_color(group, bar_color):
        if isinstance(bar_color, dict):
            return bar_color.get(group, '#CCCCCC')
        return bar_color

    xlabel = 'Interactions per 100 cells' if cell_count_df is not None else 'Interaction Count'

    if follicle_list is None:
        # ===== 只画总体 =====
        lr_summary, plot_col = get_lr_summary(pair_df, top_n, cell_count_df, 'All')
        fs = figsize if figsize else (8, 6)
        fig, ax = plt.subplots(figsize=fs)
        color = get_color('All', bar_color)
        ax.barh(range(len(lr_summary)), lr_summary[plot_col], color=color)
        ax.set_yticks(range(len(lr_summary)))
        ax.set_yticklabels(lr_summary['pair'], fontsize=10)
        ax.set_xlabel(xlabel)
        ax.set_title(f'Top {top_n} L-R Pairs: {from_leiden} → {to_leiden}')
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        return lr_summary.sort_values(plot_col, ascending=False)

    else:
        # ===== 按传入的 follicle_list 分组 =====
        all_groups = ['All'] + follicle_list
        n_plots = len(all_groups)
        ncols = 3
        nrows = -(-n_plots // ncols)

        fs = figsize if figsize else (ncols * 7, nrows * 5)
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=fs)
        axes = axes.flatten()

        for i, group in enumerate(all_groups):
            ax = axes[i]
            data = pair_df if group == 'All' else pair_df[pair_df['follicle'] == group]
            title = f'All | {from_leiden}→{to_leiden}' if group == 'All' else f'{group} | {from_leiden}→{to_leiden}'

            if len(data) == 0:
                ax.set_title(f'{title}\n(no data)')
                ax.axis('off')
                continue

            summary, plot_col = get_lr_summary(data, top_n, cell_count_df, group)
            color = get_color(group, bar_color)
            ax.barh(range(len(summary)), summary[plot_col], color=color)
            ax.set_yticks(range(len(summary)))
            ax.set_yticklabels(summary['pair'], fontsize=8)
            ax.set_xlabel(xlabel, fontsize=9)
            ax.set_title(title, fontsize=10)

        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(f'L-R Pairs by Follicle: {from_leiden} → {to_leiden}', fontsize=14)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()




In [ ]:
visualize_lr_network(
    df=df,
    from_leiden='13',
    to_leiden='3',
    top_n=10,
    follicle_list=['oo1', 'oo2', 'sf', 'ea'],
    bar_color={
        'oo1': '#B8CCE8',
        'oo2':  '#E8C8C8',
        'sf':  '#E8D8B8',
        'ea': '#D4E8B8',
    
    },
    figsize=(6, 5),
    cell_count_df=ct,
    save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 4\cellnest\3_13\lr_by_follicle_3_3.pdf'
)

In [ ]:
visualize_lr_network(
    df=df,
    from_leiden='3',
    to_leiden='13',
    top_n=10,
    follicle_list=['oo1', 'oo2', 'sf', 'ea'],
    bar_color={
        'oo1': '#B8CCE8',
        'oo2':  '#E8C8C8',
        'sf':  '#E8D8B8',
        'ea': '#D4E8B8',
    
    },
    figsize=(6, 5),
    cell_count_df=ct,
    save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 4\cellnest\3_13\lr_by_follicle_3_3.pdf'
)

In [ ]:
visualize_lr_network(
    df=df,
    from_leiden='3',
    to_leiden='3',
    top_n=10,
    follicle_list=['oo1', 'oo2', 'sf', 'ea'],
    bar_color={
        'oo1': '#B8CCE8',
        'oo2':  '#E8C8C8',
        'sf':  '#E8D8B8',
        'ea': '#D4E8B8',
    
    },
    figsize=(6, 5),
    cell_count_df=ct,
    save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 4\cellnest\3_13\lr_by_follicle_3_3.pdf'
)

# 6 Fig. 5 

## 6.1 Fig5 b-d

In [ ]:
ovary=sc.read_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adulit_ovary_raw_with_cluster_follicle_type.h5ad")

follicle=subset.copy()

In [ ]:
import pandas as pd

df = pd.read_csv(
    r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 5\From_xenium\selected_cell\ecl.csv",
    skiprows=2
)

ovary.obs['cl_type'] = 'other'


matched = ovary.obs['cell_id'].isin(df['Cell ID'])
ovary.obs.loc[matched, 'cl_type'] = 'ecl'

print(ovary.obs['cl_type'].value_counts())
print(f"matched cells: {matched.sum()}")

In [ ]:
import pandas as pd
import os

folder = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 5\From_xenium\selected_cell"


ovary.obs['cl_type'] = 'other'


for fname in os.listdir(folder):
    if not fname.endswith('.csv'):
        continue
    label = os.path.splitext(fname)[0]
    df = pd.read_csv(os.path.join(folder, fname), skiprows=2)
    matched = ovary.obs['cell_id'].isin(df['Cell ID'])
    ovary.obs.loc[matched, 'cl_type'] = label
    print(f"{label}: {matched.sum()} cells matched")

print(ovary.obs['cl_type'].value_counts())

In [ ]:
ovary.obs['cl_type'] = ovary.obs['cl_type'].astype('object')
ovary.obs['cl_type'] = ovary.obs['cl_type'].replace({'ca_1': 'ca', 'ca_2': 'ca', 'ca_3': 'ca'})
ovary.obs['cl_type'] = ovary.obs['cl_type'].astype('category')
print(ovary.obs['cl_type'].value_counts())

In [ ]:
follicle = ovary[~ovary.obs['cl_type'].isin(['other'])].copy()
follicle.obs['cl_type'].value_counts()
follicle = follicle[~follicle.obs['leiden_1'].isin(['11', '12'])].copy()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import re
import pandas as pd

color_dict = {
    "0": "#00ff7f",
    "1": "#ffd700",
    "2": "#ff8c00",
    "3": "#00FFFF",
    "4": "#ff00ff",
    "5": "#32cd32",
    "6": "#00ced1",
    "7": "#ff4500",
    "8": "#ff69b4",
    "9": "#ff002a",
    "10": "#565dfd",
    "11": "#585b5c",
    "12": "#1e90ff",
    "13": "#ADFF2F",
}

def sort_key(s):
    match = re.match(r'([a-zA-Z]+)(\d+)', s)
    if match:
        return (match.group(1), int(match.group(2)))

    if s.isdigit():
        return ('', int(s))

    return (s, 0)


def plot_follicle_composition(
    sub_adata,
    order=None,
    title=None,
    save_path=None
):
    sub = sub_adata.obs

    if order is None:
        order = ['ecl', 'mcl', 'lcl', 'ca']

    data, totals = {}, {}

    for ft in order:
        group = sub[
            sub['cl_type'] == ft
        ]['leiden_1']

        totals[ft] = len(group)
        data[ft] = group.value_counts(
            normalize=True
        )

    df_plot = (
        pd.DataFrame(data)
        .fillna(0)
        .T
        .reindex(order)
    )

    all_merge = sorted(
        df_plot.columns.tolist(),
        key=sort_key
    )

    df_plot = df_plot[all_merge]

    fig, ax = plt.subplots(
        figsize=(5, 4.7)
    )

    bottom = np.zeros(
        len(order)
    )

    for col in reversed(all_merge):
        vals = df_plot[col].values
        color = color_dict.get(
            col,
            '#CCCCCC'
        )

        ax.bar(
            order,
            vals,
            bottom=bottom,
            label=col,
            color=color,
            width=0.95
        )

        bottom += vals

    for i, ft in enumerate(order):
        ax.text(
            i,
            1.01,
            f'n={totals[ft]}',
            ha='center',
            va='bottom',
            fontsize=9
        )

    ax.set_ylabel('Proportion')
    ax.set_xlabel('follicle_type_new')
    ax.set_ylim(0, 1.1)

    if title:
        ax.set_title(title)

    handles, labels = (
        ax.get_legend_handles_labels()
    )

    ax.legend(
        handles[::-1],
        labels[::-1],
        bbox_to_anchor=(1.01, 1),
        loc='upper left',
        fontsize=8
    )

    plt.tight_layout()

    if save_path:
        plt.savefig(
            save_path,
            bbox_inches='tight'
        )

    plt.show()

In [ ]:
sub = follicle
plot_follicle_composition(sub,save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 5\quantification\cl_leiden_1.pdf')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import re
import pandas as pd

gc_colors = {
    f'gc{i}': c
    for i, c in enumerate([
        '#E6194B', '#3CB44B', '#FFE119', '#4363D8', '#F032E6',
        '#911EB4', '#42D4F4', '#F58231', '#BFEF45', '#FABEBE',
        '#469990', '#E6BEFF', '#9A6324'
    ])
}

tc_colors = {
    f'tc{i}': c
    for i, c in enumerate([
        '#00FFFF', '#800000', '#AAFFC3', '#808000', '#FFDAC1',
        '#000075', '#A9A9A9', '#DC143C', '#FFFAC8', '#FF1493',
        '#7FFFD4'
    ])
}

bl_colors = {
    f'bl{i}': c
    for i, c in enumerate([
        '#FF4500', '#32CD32', '#1E90FF', '#FFD700', '#8A2BE2',
        '#00FA9A', '#FF69B4', '#00CED1', '#FF8C00', '#9932CC'
    ])
}

im_colors = {
    f'im{i}': c
    for i, c in enumerate([
        '#7FFF00', '#4682B4', '#B22222', '#BA55D3', '#6495ED',
        '#DAA520', '#C71585', '#20B2AA', '#F4A460', '#40E0D0',
        '#FF6347', '#00FF7F', '#2E8B57'
    ])
}

color_dict = {
    **gc_colors,
    **tc_colors,
    **bl_colors,
    **im_colors
}


def sort_key(s):
    match = re.match(r'([a-zA-Z]+)(\d+)', s)

    if match:
        return (
            match.group(1),
            int(match.group(2))
        )

    if s.isdigit():
        return (
            '',
            int(s)
        )

    return (
        s,
        0
    )


def plot_follicle_composition(
    sub_adata,
    order=None,
    title=None,
    save_path=None
):
    sub = sub_adata.obs

    if order is None:
        order = [
            'ecl',
            'mcl',
            'lcl',
            'ca'
        ]

    data = {}
    totals = {}

    for ft in order:
        group = sub[
            sub['cl_type'] == ft
        ]['merge']

        totals[ft] = len(group)

        data[ft] = group.value_counts(
            normalize=True
        )

    df_plot = (
        pd.DataFrame(data)
        .fillna(0)
        .T
        .reindex(order)
    )

    all_merge = sorted(
        df_plot.columns.tolist(),
        key=sort_key
    )

    df_plot = df_plot[
        all_merge
    ]

    fig, ax = plt.subplots(
        figsize=(5, 4.7)
    )

    bottom = np.zeros(
        len(order)
    )

    for col in reversed(all_merge):

        vals = df_plot[col].values

        color = color_dict.get(
            col,
            '#CCCCCC'
        )

        ax.bar(
            order,
            vals,
            bottom=bottom,
            label=col,
            color=color,
            width=0.95
        )

        bottom += vals

    for i, ft in enumerate(order):

        ax.text(
            i,
            1.01,
            f'n={totals[ft]}',
            ha='center',
            va='bottom',
            fontsize=9
        )

    ax.set_ylabel(
        'Proportion'
    )

    ax.set_xlabel(
        'follicle_type_new'
    )

    ax.set_ylim(
        0,
        1.1
    )

    if title:
        ax.set_title(title)

    handles, labels = (
        ax.get_legend_handles_labels()
    )

    ax.legend(
        handles[::-1],
        labels[::-1],
        bbox_to_anchor=(1.01, 1),
        loc='upper left',
        fontsize=8
    )

    plt.tight_layout()

    if save_path:
        plt.savefig(
            save_path,
            bbox_inches='tight'
        )

    plt.show()

In [ ]:
sub = follicle[follicle.obs['leiden_1'] == '4']
plot_follicle_composition(sub,save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 5\quantification\cl_bl.pdf')

In [ ]:
sub = follicle[follicle.obs['leiden_1'] == '0']
plot_follicle_composition(sub,save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 5\quantification\cl_im.pdf')

In [ ]:
# 

In [ ]:
ovary=sc.read_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adulit_ovary_raw_with_cluster_follicle_type.h5ad")

follicle=subset.copy()

# 7 Fig. 6 

## 7.1 Fig. 6b 

In [ ]:
import scanpy as sc

file_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\Analysis\5st_2025_07_22_for paper\adult_voary\adulit_ovary_raw_with_cluster_follicle_type.h5ad"

ovary = sc.read_h5ad(file_path)

In [ ]:
adata.obs['age'] = adata.obs['roi'].str.strip('_').str.split('_').str[-1]
print(adata.obs['age'].value_counts())

In [ ]:
adata.obs['age_numeric'] = pd.to_numeric(adata.obs['age'], errors='coerce')
adata.obs['age_numeric'] = adata.obs['age_numeric'].replace(28, 68)
def assign_age_group(row):
    age = row['age_numeric']
    if pd.isna(age):
        return None
    if age <= 4:
        return '4 yr'
    elif age <= 12:
        return '7-12 yr'
    elif age <= 18:
        return '13-18 yr'
    elif age <= 32:
        return '18-32 yr'
    elif age <= 60:
        return '>=50 yr'
    else:
        return '60'

adata.obs['age_group'] = adata.obs.apply(assign_age_group, axis=1)
print(adata.obs['age_group'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import re
import pandas as pd

color_dict_k27 = {
    '0':  '#FFD8B1',
    '1':  '#3CB44B',
    '2':  '#FFE119',
    '3':  '#4363D8',
    '4':  '#FFFAC8',
    '5':  '#911EB4',
    '6':  '#1E90FF',
    '7':  '#00CED1',
    '8':  '#BFEF45',
    '9':  '#FABEBE',
    '10': '#469990',
    '11': '#556B2F',
    '12': '#9A6324',
    '13': '#E6194B',
    '14': '#800000',
    '15': '#FF69B4',
    '16': '#808000',
    '17': '#F58231',
    '18': '#000075',
    '19': '#A9A9A9',
    '20': '#FF1493',
    '21': '#F032E6',
    '22': '#42D4F4',
    '23': '#7FFFD4',
    '24': '#556B2F',
    '25': '#00FA9A',
    '26': '#AAFFC3',
}

def sort_key_cluster(s):
    if s.isdigit():
        return ('', int(s))
    match = re.match(r'([a-zA-Z]+)(\d+)', s)
    if match:
        return (match.group(1), int(match.group(2)))
    return (s, 0)

def plot_age_composition(sub_adata, order=None, title=None, save_path=None, split_group='>=50 yr', gap=0.5):
    sub = sub_adata.obs
    if order is None:
        order = ['4 yr', '7-12 yr', '13-18 yr', '18-32 yr', '>=50 yr', '60']

    data, totals = {}, {}
    for age_grp in order:
        group = sub[sub['age_group'] == age_grp]['cluster_cellcharter_k27']
        totals[age_grp] = len(group)
        data[age_grp] = group.value_counts(normalize=True)

    df_plot = pd.DataFrame(data).fillna(0).T.reindex(order)
    all_clusters = sorted(df_plot.columns.tolist(), key=sort_key_cluster)
    df_plot = df_plot[all_clusters]

    
    x_positions = []
    for i, grp in enumerate(order):
        if split_group and order.index(grp) > order.index(split_group):
            x_positions.append(i + gap)
        else:
            x_positions.append(float(i))

    fig, ax = plt.subplots(figsize=(5, 5))
    bottom = np.zeros(len(order))

    for col in reversed(all_clusters):
        vals = df_plot[col].values
        color = color_dict_k27.get(col, '#CCCCCC')
        ax.bar(x_positions, vals, bottom=bottom, label=col, color=color, width=0.95)
        bottom += vals

    for i, age_grp in enumerate(order):
        ax.text(x_positions[i], 1.01, f'n={totals[age_grp]}', ha='center', va='bottom', fontsize=9)

  
    if split_group and split_group in order:
        idx = order.index(split_group)
        ax.axvline(x=x_positions[idx] + 0.5 + gap / 2, color='black', linestyle='--', linewidth=1.5)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(order, rotation=0)
    ax.set_ylabel('Proportion')
    ax.set_xlabel('Age Group')
    ax.set_ylim(0, 1.1)
    if title:
        ax.set_title(title)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()

plot_age_composition(
    adata,
    split_group='>=50 yr',
    gap=0.5,
    save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 6\bar_plot\age_composition_k27_agegroup.pdf'
)

In [ ]:
## 7.1 Fig. 6c

In [ ]:
cc.gr.enrichment(adata, group_key='cluster_cellcharter_k27', label_key='leiden_1')
cc.pl.enrichment(
    adata,
    group_key='cluster_cellcharter_k27',
    label_key='leiden_1',
    figsize=(6, 3),
    dot_scale=1,
    save=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 6\enrichment\enrichment_k27_leiden_1.pdf'
)

## 7.1 Fig. 6d 

In [ ]:
import gseapy as gp
import scanpy as sc

hallmark = gp.get_library(name='MSigDB_Hallmark_2020', organism='Human')


for pathway_name, gene_list in hallmark.items():
    score_key = pathway_name.replace(' ', '_') + '_Score'
    genes_in_adata = [g for g in gene_list if g in adata.var_names]
    if len(genes_in_adata) < 5:
        continue
    sc.tl.score_genes(
        adata,
        gene_list=genes_in_adata,
        score_name=score_key
    )
    print(f"✓ {pathway_name}: {len(genes_in_adata)} gene")


score_columns = [col for col in adata.obs.columns if '_Score' in col]


In [ ]:
xenium_genes = set(adata.var_names)

coverage_dict = {}

for col in score_columns:
    pathway_name = (
        col
        .replace('_Score', '')
        .replace('_', ' ')
    )

    gene_list = hallmark.get(pathway_name, [])

    if len(gene_list) == 0:
        print(f"Warning: pathway not found: {pathway_name}")
        continue

    total = len(gene_list)
    found = len([
        g for g in gene_list
        if g in xenium_genes
    ])

    coverage = found / total * 100
    coverage_dict[col] = coverage

filtered_columns = [
    col
    for col, cov in coverage_dict.items()
    if cov >= 40
]

print(f"Original number of pathways: {len(score_columns)}")
print(f"Number of pathways with coverage >= 40%: {len(filtered_columns)}")

In [ ]:
xenium_genes = set(adata.var_names)

coverage_dict = {}
for col in score_columns:
    pathway_name = col.replace('_Score', '').replace('_', ' ')
    gene_list = hallmark.get(pathway_name, [])
    if len(gene_list) == 0:
       
        continue
    total = len(gene_list)
    found = len([g for g in gene_list if g in xenium_genes])
    coverage = found / total * 100
    coverage_dict[col] = coverage

filtered_columns = [col for col, cov in coverage_dict.items() if cov >= 40]


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage, leaves_list
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import set_link_color_palette

fig = plt.figure(figsize=(16, 12))

df_mean = adata_temp.to_df()
df_mean['cluster'] = adata_temp.obs['cluster_cellcharter_k27'].values
mean_mat = df_mean.groupby('cluster').mean()

df_scaled = pd.DataFrame(
    StandardScaler().fit_transform(mean_mat),
    index=mean_mat.index,
    columns=mean_mat.columns
)

col_linkage = linkage(
    pdist(df_scaled, metric='euclidean'),
    method='ward'
)

sorted_cluster_idx = leaves_list(col_linkage)

sorted_clusters = [
    df_scaled.index[i]
    for i in sorted_cluster_idx
]

sorted_pathways = df_scaled.columns.tolist()

mean_scaled = df_scaled.loc[
    sorted_clusters,
    sorted_pathways
]

frac_mat = df_mean.groupby('cluster')[sorted_pathways].apply(
    lambda x: (x > 0).mean()
)

frac_mat = frac_mat.loc[
    sorted_clusters,
    sorted_pathways
]

fig = plt.figure(figsize=(12, 12))

gs = gridspec.GridSpec(
    2,
    1,
    height_ratios=[1, 8],
    hspace=0.02
)

ax_col_dendro = fig.add_subplot(gs[0])

set_link_color_palette(['black'])

dendrogram(
    col_linkage,
    orientation='top',
    labels=sorted_clusters,
    ax=ax_col_dendro,
    color_threshold=0,
    above_threshold_color='black'
)

ax_bubble = fig.add_subplot(gs[1])

for yi, pathway in enumerate(sorted_pathways):
    for xi, cluster in enumerate(sorted_clusters):

        color_val = mean_scaled.loc[
            cluster,
            pathway
        ]

        size_val = frac_mat.loc[
            cluster,
            pathway
        ]

        ax_bubble.scatter(
            xi,
            yi,
            s=size_val * 300,
            c=color_val,
            cmap='RdBu_r',
            vmin=-2,
            vmax=2,
            alpha=0.9,
            linewidths=0
        )

ax_bubble.set_xticks(
    range(len(sorted_clusters))
)

ax_bubble.set_xticklabels(
    sorted_clusters,
    rotation=90,
    fontsize=8
)

ax_bubble.set_yticks(
    range(len(sorted_pathways))
)

ax_bubble.set_yticklabels(
    sorted_pathways,
    fontsize=8
)

ax_bubble.grid(False)

sm = plt.cm.ScalarMappable(
    cmap='RdBu_r',
    norm=plt.Normalize(
        vmin=-2,
        vmax=2
    )
)

sm.set_array([])

plt.colorbar(
    sm,
    ax=ax_bubble,
    label='Relative Activity',
    shrink=0.5,
    pad=0.02
)

for size in [0.25, 0.5, 0.75, 1.0]:

    ax_bubble.scatter(
        [],
        [],
        s=size * 300,
        c='grey',
        alpha=0.6,
        label=f'{int(size * 100)}%'
    )

ax_bubble.legend(
    title='Fraction of Cells',
    bbox_to_anchor=(1.15, 1),
    loc='upper left',
    fontsize=8
)

save_path = (
    r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project'
    r'\19_Xenium 5K_prepurvty ovary\paper\figure'
    r'\1st_2026_03_19\Figure 6\pathway'
    r'\HALLMARK_pathway_k27_bubble.pdf'
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
## 7.1 Fig. 6e 

In [ ]:
cortex1 = cortex[cortex.obs['leiden_1'].isin([ "5",'8', '9', '10','6','7'])].copy()
print(cortex1)

In [ ]:
heatmap_genes = [
    'NR4A1',      
    'ADAMTS1',   
    'SGK1',       
    'EEF1G',      
    'TIMP2',      
    'CCN5',      
    'FOXO1',      
    'TUBB',      
    'CD22',       
    'HNRNPH1',    
    'THBS1',      
    'ENG',        
    'NBL1',       
    'TCF21',      
    'PTK7',     
    'MSI2',       
    'STAR',       
    'PGR',        
    'FOXL2',      
    'IGFBP2',    
]

sc.pl.dotplot(
    cortex1,
    var_names=heatmap_genes,
    groupby='leiden_1',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(7, 2),
    dendrogram=True,
    save='stroma_leiden_1.1_normalize_2ST.pdf'
)

## 7.1 Fig. 6f 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import re
import pandas as pd

gc_colors = {f'gc{i}': c for i, c in enumerate([
    '#E6194B','#3CB44B','#FFE119','#4363D8','#F032E6','#911EB4',
    '#42D4F4','#F58231','#BFEF45','#FABEBE','#469990','#E6BEFF','#9A6324'
])}
tc_colors = {f'tc{i}': c for i, c in enumerate([
    '#00FFFF','#800000','#AAFFC3','#808000','#FFDAC1','#000075',
    '#A9A9A9','#DC143C','#FFFAC8','#FF1493','#7FFFD4'
])}
bl_colors = {f'bl{i}': c for i, c in enumerate([
    '#FF4500','#32CD32','#1E90FF','#FFD700','#8A2BE2','#00FA9A',
    '#FF69B4','#00CED1','#FF8C00','#9932CC'
])}
im_colors = {f'im{i}': c for i, c in enumerate([
    '#7FFF00','#4682B4','#B22222','#BA55D3','#6495ED','#DAA520',
    '#C71585','#20B2AA','#F4A460','#40E0D0','#FF6347','#00FF7F','#2E8B57'
])}
color_dict = {**gc_colors, **tc_colors, **bl_colors, **im_colors}

def sort_key(s):
    match = re.match(r'([a-zA-Z]+)(\d+)', s)
    if match:
        return (match.group(1), int(match.group(2)))
    if s.isdigit():
        return ('', int(s))
    return (s, 0)

def plot_follicle_composition(sub_adata, order=None, title=None, save_path=None, split_cluster=None, gap=0.5):
    sub = sub_adata.obs
    if order is None:
        order = ['9', '0', '4', '8', '5', '10', '7', '24', '23', '18']

    data, totals = {}, {}
    for ft in order:
        group = sub[sub['cluster_cellcharter_k27'] == ft]['merge']
        totals[ft] = len(group)
        data[ft] = group.value_counts(normalize=True)

    df_plot = pd.DataFrame(data).fillna(0).T.reindex(order)
    all_merge = sorted(df_plot.columns.tolist(), key=sort_key)
    df_plot = df_plot[all_merge]

   
    x_positions = []
    for i, cluster in enumerate(order):
        if split_cluster and order.index(cluster) > order.index(split_cluster):
            x_positions.append(i + gap)
        else:
            x_positions.append(float(i))

    fig, ax = plt.subplots(figsize=(7, 4.7))
    bottom = np.zeros(len(order))

    for col in reversed(all_merge):
        vals = df_plot[col].values
        color = color_dict.get(col, '#CCCCCC')
        ax.bar(x_positions, vals, bottom=bottom, label=col, color=color, width=0.95)
        bottom += vals

    for i, ft in enumerate(order):
        ax.text(x_positions[i], 1.01, f'n={totals[ft]}', ha='center', va='bottom', fontsize=9)

  
    if split_cluster and split_cluster in order:
        idx = order.index(split_cluster)
        ax.axvline(x=x_positions[idx] + 0.5 + gap / 2, color='black', linestyle='--', linewidth=1.5)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(order, rotation=0)
    ax.set_ylabel('Proportion')
    ax.set_xlabel('Cluster')
    ax.set_ylim(0, 1.1)
    if title:
        ax.set_title(title)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()

plot_follicle_composition(
    sub,
    split_cluster='10',
    gap=0.5, save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\19_Xenium 5K_prepurvty ovary\paper\figure\1st_2026_03_19\Figure 6\bar_plot\cortex_medulla_composition_k27_leiden_1_im.pdf'
)